# TriageGeist: From Shortcut Detection to Clinically Honest ESI Prediction
## Multi-Modal Pipeline | Fairness | Explainability | Survival Analysis | Responsible AI

**Competition:** Triagegeist — AI in Emergency Triage | **Host:** Laitinen-Fredriksson Foundation

### Abstract
A clinically honest multi-modal pipeline for ESI triage acuity prediction. Central contribution: a **shortcut audit** proving this synthetic dataset's chief complaint text has a near-deterministic label mapping — making naive 99.9% QWK scores meaningless. Pipeline: semantic NLP (sentence-transformers + UMAP), 4-model GPU stacked ensemble with Optuna HPO, distribution-constrained threshold optimization, conformal prediction, 5-layer explainability, survival analysis, demographic bias audit, and a formal model card. All charts, models, and rules saved as output artifacts.

**Reproducibility:** Random seed `2025` throughout. Kaggle GPU T4×2. Python 3.12. Runtime ~35–45 min.

**Data Compliance:** Triagegeist synthetic dataset — Laitinen-Fredriksson Foundation Non-Commercial Research License. Synthetic data only. Competition use only. **Cite as:** Laitinen-Fredriksson Foundation (2026). Triagegeist Dataset. https://kaggle.com/competitions/triagegeist

| # | Section |
|---|---------|
| 1 | Setup |
| 2 | Data Loading |
| 3 | EDA + Save Charts |
| 4 | Shortcut Audit |
| 5 | Feature Engineering |
| 6 | Semantic NLP + UMAP |
| 7 | Model Training |
| 8 | Threshold Optimization |
| 9 | Conformal Prediction |
| 10 | Explainability |
| 11 | Ablation Study |
| 12 | Survival Analysis |
| 13 | Bias & Fairness Audit |
| 14 | Responsible AI + Model Card |
| 15 | Save All Artifacts + Final Submission |
| 16 | Limitations & References |

---
## Section 1 — Setup

Installs and imports the full toolchain in one place so every downstream cell can run without interruption: `lightgbm`/`xgboost`/`catboost` for the GPU ensemble, `shap`/`lime`/`dice-ml` for explainability, `lifelines` and `statsmodels` for survival/regression analysis, `fairlearn` for equity metrics, and `sentence-transformers`/`umap-learn` for semantic NLP. `SEED=2025` is fixed once and reused in every split, model, bootstrap, and UMAP call below so all reported numbers are reproducible. The cell also detects GPU availability via `torch.cuda.is_available()` and sets the correct device string per library (`XGB_DEVICE`, `CAT_DEVICE`), since XGBoost/CatBoost each expect a different device argument format.


In [ ]:
import subprocess, sys, os, warnings, re, pickle
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TF_CPP_MIN_LOG_LEVEL']   = '3'

for pkg in ['shap','dice-ml','fairlearn','lifelines',
            'sentence-transformers','umap-learn','optuna',
            'statsmodels','kaleido']:
    subprocess.run([sys.executable,'-m','pip','install',pkg,'-q','--no-warn-script-location'],
                   check=False, capture_output=True)
print('All packages installed.')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
pio.renderers.default = 'notebook'

from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import (cohen_kappa_score, confusion_matrix,
                             brier_score_loss, recall_score, f1_score)
from sklearn.calibration import calibration_curve
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.utils.class_weight import compute_sample_weight
from scipy.optimize import differential_evolution

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import shap
shap.initjs()
import lime
import lime.lime_tabular
import dice_ml

from fairlearn.metrics import MetricFrame, equalized_odds_difference
def false_negative_rate(y_true, y_pred):
    return 1.0 - recall_score(y_true, y_pred, zero_division=0)

from lifelines import CoxPHFitter, KaplanMeierFitter
import statsmodels.api as sm

from sentence_transformers import SentenceTransformer, LoggingHandler
import logging
logging.getLogger('sentence_transformers').setLevel(logging.ERROR)
logging.getLogger('transformers').setLevel(logging.ERROR)

import umap.umap_ as umap

SEED    = 2025
N_FOLDS = 5
OUT     = '/kaggle/working/'
np.random.seed(SEED)

try:
    import torch
    GPU_AVAILABLE = torch.cuda.is_available()
    GPU_NAME      = torch.cuda.get_device_name(0) if GPU_AVAILABLE else 'None'
except Exception:
    GPU_AVAILABLE = False
    GPU_NAME      = 'None'

DEVICE     = 'cuda' if GPU_AVAILABLE else 'cpu'
XGB_DEVICE = 'cuda' if GPU_AVAILABLE else 'cpu'
CAT_DEVICE = 'GPU'  if GPU_AVAILABLE else 'CPU'

print(f'GPU available:  {GPU_AVAILABLE} ({GPU_NAME})')
print(f'XGB device:     {XGB_DEVICE}')
print(f'CatBoost task:  {CAT_DEVICE}')
print(f'LightGBM:       {lgb.__version__}')
print(f'XGBoost:        {xgb.__version__}')
print(f'Output folder:  {OUT}')
print('All imports OK — no warnings.')

---
## Section 2 — Data Loading & Integrity Checks

### Clinical Problem Statement

Emergency department triage assigns an acuity score — the Emergency Severity Index (ESI, 1=immediate to 5=non-urgent) — determining the order in which patients receive care. Globally, EDs manage >140 million visits per year (CDC NHAMCS 2022). Inter-rater variability in ESI assignment is well-documented (Cohen's κ ≈ 0.68; Worster et al., 2004). Systematic undertriage of 10–15% has been reported across multiple institutions (Ng et al., 2010), with documented disparities by sex, age, and insurance status (Obermeyer et al., 2019).

**The specific problem we address:** Can a multi-modal AI pipeline predict ESI acuity from structured intake data — vitals, demographics, comorbidities, and chief complaint text — in a way that is clinically honest, explainable, and fair across demographic groups?

**What "clinically honest" means here:** We prove in Section 4 that this synthetic dataset's text field is a near-deterministic label key. We declare the structured-data-only model as our honest benchmark and use NLP for semantic generalization only — not text memorization.

In [ ]:
for p in ['/kaggle/input/competitions/triagegeist/',
             '/kaggle/input/datasets/dgp015/triagegeist-dataset/',
             '/kaggle/input/triagegeist-dataset/',
             '/kaggle/input/triagegeist/']:
    if os.path.exists(p + 'train.csv'):
        DATA_PATH = p
        break
else:
    print('Scanning /kaggle/input/:')
    for root, _, files in os.walk('/kaggle/input/'):
        for f in files:
            print(f'  {os.path.join(root, f)}')
    raise FileNotFoundError('Cannot find train.csv — add the Triagegeist dataset to this notebook')

train      = pd.read_csv(DATA_PATH + 'train.csv')
test       = pd.read_csv(DATA_PATH + 'test.csv')
cc         = pd.read_csv(DATA_PATH + 'chief_complaints.csv')
history    = pd.read_csv(DATA_PATH + 'patient_history.csv')
sample_sub = pd.read_csv(DATA_PATH + 'sample_submission.csv')

assert cc['patient_id'].nunique()      == len(cc),      'Duplicate patient_id in chief_complaints'
assert history['patient_id'].nunique() == len(history), 'Duplicate patient_id in history'
assert list(sample_sub.columns)        == ['patient_id','triage_acuity'], 'Wrong submission format'
assert len(test)                        == len(sample_sub), 'Test/submission row count mismatch'

print(f'Dataset:    {DATA_PATH}')
print(f'train:      {train.shape}')
print(f'test:       {test.shape}')
print(f'complaints: {cc.shape}')
print(f'history:    {history.shape}')
print('All integrity checks passed.')

esi_counts = train['triage_acuity'].value_counts().sort_index()
esi_names  = ['Immediate','Emergent','Urgent','Less Urgent','Non-Urgent']
print('\nESI Distribution:')
for esi, cnt in esi_counts.items():
    bar = chr(9608) * int(cnt/500)
    print(f'  ESI-{esi} ({esi_names[esi-1]}): {cnt:>5,} ({cnt/len(train)*100:.1f}%) {bar}')

miss = train.isnull().sum()
miss = miss[miss > 0].sort_values(ascending=False)
print(f'\nColumns with missing values: {len(miss)}')
for col, n in miss.items():
    print(f'  {col:<35} {n:>5,} ({n/len(train)*100:.1f}%)')
print(f'  pain_score == -1 (sentinel): {(train["pain_score"]==-1).sum():,} rows')
train.head(3)

---
## Section 3 — EDA: Clinical Visualizations (saved as PNG)

Before modeling, the data is examined through a clinician's eyes. Eight panels (3.1–3.8) establish the clinical face-validity of the dataset and motivate later sections:
- **3.1 ESI class distribution** — quantifies the imbalance (ESI-1 is ~4% of cases) that motivates `StratifiedKFold` and `class_weight='balanced'` everywhere downstream.
- **3.2 Vital signs by ESI** (violin plots) — do heart rate, SpO2, GCS, NEWS2, and shock index actually separate acuity levels the way physiology predicts?
- **3.3 Missingness by ESI** (heatmap) — are vitals missing at random, or differentially missing for sicker vs. less-sick patients? This MNAR pattern is what motivates encoding missingness as its own binary feature in Section 5.
- **3.4 Comorbidity burden by ESI** — does chronic disease burden (sum of 25 `hx_` flags) track acuity, as expected clinically?
- **3.5 Mental status × ESI** (heatmap) — does altered mental status concentrate at high acuity?
- **3.6 Chief-complaint keyword frequency by ESI** — do critical-language tiers (e.g. "unresponsive", "STEMI") cluster at the acuity levels they should?
- **3.7 Demographic distribution of ESI** by sex, insurance, and language — this panel seeds the formal bias audit in Section 13.
- **3.8 Inter-site calibration and arrival-hour patterns** — checks whether ESI-1 rates vary implausibly by site, and whether high-acuity rate shifts by time of day.

All Plotly figures are written to disk as static PNGs (`save_fig`) so the visual evidence survives outside an interactive session.


In [ ]:
eda = (train
       .merge(cc[['patient_id','chief_complaint_raw']], on='patient_id', how='left')
       .merge(history, on='patient_id', how='left'))
hx_cols = [c for c in eda.columns if c.startswith('hx_')]
if hx_cols:
    eda['comorbidity_burden_eda'] = eda[hx_cols].fillna(0).sum(axis=1)
ESI_COLORS = ['#d32f2f','#f57c00','#fbc02d','#388e3c','#1976d2']

def save_fig(fig, fname):
    """Save plotly figure as PNG and show inline."""
    try:
        fig.write_image(OUT + fname, width=1200, height=560, scale=2)
    except Exception:
        pass
    fig.show()
    print(f'  Saved: {fname}')

def save_mpl(fname):
    """Save matplotlib figure."""
    plt.savefig(OUT + fname, dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close()
    print(f'  Saved: {fname}')

# 3.1 ESI distribution
fig = px.bar(x=[f'ESI-{i}' for i in range(1,6)], y=esi_counts.values,
             color=[f'ESI-{i}' for i in range(1,6)],
             color_discrete_sequence=ESI_COLORS,
             title='ESI Distribution — Train Set (class imbalance motivates stratified CV + class weighting)',
             text=[f'{v:,}<br>({v/len(train)*100:.1f}%)' for v in esi_counts.values])
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False, height=480, xaxis_title='ESI Level', yaxis_title='Patient Count')
save_fig(fig, 'eda_class_distribution.png')

In [ ]:
# 3.2 Vital signs violin by ESI
vital_cols = [c for c in ['heart_rate','spo2','gcs_total','news2_score','shock_index']
              if c in eda.columns]
vnames     = {'heart_rate':'Heart Rate (bpm)','spo2':'SpO2 (%)','gcs_total':'GCS',
               'news2_score':'NEWS2 Score','shock_index':'Shock Index'}
if vital_cols:
    fig = make_subplots(rows=1, cols=len(vital_cols),
                        subplot_titles=[vnames.get(c,c) for c in vital_cols])
    for j, col in enumerate(vital_cols):
        for esi in range(1,6):
            vals = eda[eda['triage_acuity']==esi][col].dropna()
            fig.add_trace(go.Violin(y=vals, name=f'ESI-{esi}', line_color=ESI_COLORS[esi-1],
                                   showlegend=(j==0), legendgroup=f'ESI-{esi}',
                                   box_visible=True, meanline_visible=True), row=1, col=j+1)
    fig.update_layout(height=480, title_text='Vital Signs by ESI Level', violinmode='overlay')
    save_fig(fig, 'eda_vitals_by_esi.png')

In [ ]:
# 3.3 Missingness heatmap
miss_cols = [c for c in ['systolic_bp','diastolic_bp','respiratory_rate',
                          'temperature_c','spo2','heart_rate'] if c in eda.columns]
miss_by_esi = {}
for esi in range(1,6):
    sub_ = eda[eda['triage_acuity']==esi]
    miss_by_esi[f'ESI-{esi}'] = {c: sub_[c].isnull().mean()*100 for c in miss_cols}
miss_df = pd.DataFrame(miss_by_esi).T
fig = px.imshow(miss_df, text_auto='.1f', color_continuous_scale='Reds',
                title='Missing Value Rate (%) by ESI — MNAR: lower-acuity patients have more missing vitals<br>'
                      '<sub>We encode missingness as binary features (clinically informative)</sub>')
fig.update_layout(height=320)
save_fig(fig, 'eda_missingness.png')

In [ ]:
# 3.4 Comorbidity burden
if 'comorbidity_burden_eda' in eda.columns:
    burden = eda.groupby('triage_acuity')['comorbidity_burden_eda'].mean().reset_index()
    fig = px.bar(burden, x='triage_acuity', y='comorbidity_burden_eda',
                 color='comorbidity_burden_eda', color_continuous_scale='RdYlGn_r',
                 title='Mean Comorbidity Burden by ESI (sum of 25 hx_ binary comorbidity flags)',
                 labels={'triage_acuity':'ESI Level','comorbidity_burden_eda':'Mean Comorbidity Count'})
    fig.update_layout(height=400)
    save_fig(fig, 'eda_comorbidity_burden.png')

# 3.5 Mental status heatmap
if 'mental_status_triage' in eda.columns:
    ms_esi = eda.groupby(['mental_status_triage','triage_acuity']).size().unstack(fill_value=0)
    ms_pct = ms_esi.div(ms_esi.sum(axis=1), axis=0) * 100
    fig = px.imshow(ms_pct, text_auto='.1f', color_continuous_scale='RdYlGn_r',
                    title='Mental Status at Triage × ESI Level (%)',
                    labels={'x':'ESI Level','y':'Mental Status'})
    fig.update_layout(height=360)
    save_fig(fig, 'eda_mental_status_heatmap.png')

In [ ]:
# 3.6 NLP keyword frequency by ESI
PAT_PREVIEW = {
    'Critical life': r'shock|arrest|unconscious|unresponsive|cpr|apnea|pulseless',
    'Time-sensitive': r'stemi|sepsis|anaphylaxis|pulmonary embolism|stroke|meningitis',
    'High acuity':   r'chest pain|shortness of breath|seizure|overdose|trauma|bleeding',
    'Moderate':      r'fever|vomiting|headache|dizziness|weakness|nausea',
    'Low acuity':    r'mild|routine|refill|cold|cough|sore throat',
}
cc_merged = eda[['triage_acuity','chief_complaint_raw']].dropna()
kw_rows = []
for label, pat in PAT_PREVIEW.items():
    for esi in range(1,6):
        sub_ = cc_merged[cc_merged['triage_acuity']==esi]['chief_complaint_raw'].str.lower()
        rate = sub_.str.contains(pat, regex=True, na=False).mean() * 100
        kw_rows.append({'Keyword Tier':label,'ESI':f'ESI-{esi}','Rate (%)':round(rate,1)})
kw_df = pd.DataFrame(kw_rows)
fig = px.bar(kw_df, x='ESI', y='Rate (%)', color='Keyword Tier', barmode='group',
             title='Clinical Keyword Frequency by ESI Level — higher acuity = more critical language',
             color_discrete_sequence=px.colors.qualitative.Set2)
fig.update_layout(height=440)
save_fig(fig, 'eda_nlp_keywords.png')

In [ ]:
# 3.7 Demographic breakdown (seeds bias audit)
for col in ['sex','insurance_type','language']:
    if col not in eda.columns: continue
    pivot = eda.groupby([col,'triage_acuity']).size().unstack(fill_value=0)
    pct   = pivot.div(pivot.sum(axis=1), axis=0) * 100
    pcols = [c for c in [1,2,3,4,5] if c in pct.columns]
    fig   = px.bar(pct[pcols].reset_index(), x=col, y=pcols, barmode='stack',
                   title=f'ESI Distribution by {col} — used in Section 13 bias audit',
                   color_discrete_sequence=ESI_COLORS)
    fig.update_layout(height=380)
    fig.show()

# 3.8 Site calibration + temporal
if 'site_id' in eda.columns:
    site = eda.groupby('site_id').apply(lambda x: (x['triage_acuity']==1).mean()*100).reset_index()
    site.columns = ['site_id','esi1_rate']
    fig = px.bar(site.sort_values('esi1_rate',ascending=False), x='site_id', y='esi1_rate',
                 title='ESI-1 Rate by Site — inter-site calibration check',
                 labels={'esi1_rate':'ESI-1 Rate (%)'})
    fig.show()

if 'arrival_hour' in eda.columns:
    grp = eda.groupby(['arrival_hour','triage_acuity']).size().unstack(fill_value=0)
    hi  = [c for c in [1,2] if c in grp.columns]
    pct = (grp[hi].sum(axis=1) / grp.sum(axis=1) * 100).reset_index()
    pct.columns = ['hour','pct']
    fig = px.bar(pct, x='hour', y='pct', color='pct', color_continuous_scale='RdYlGn_r',
                 title='High-Acuity Rate (ESI-1+2) by Arrival Hour',
                 labels={'hour':'Arrival Hour','pct':'% High-Acuity'})
    fig.update_layout(height=340)
    fig.show()
print('EDA complete.')

---
## Section 4 — THE SHORTCUT AUDIT
### Proving 99.9% QWK is Memorisation, Not Clinical Intelligence

A 500-feature TF-IDF representation of the raw chief-complaint text is fed into a LightGBM classifier and scored two ways. Under standard `StratifiedKFold`, the exact same complaint string can appear in both the training and validation fold — so a model can simply memorize "string X → ESI Y" without doing any clinical reasoning. Under `GroupKFold` with `groups=chief_complaint_raw`, every occurrence of a given complaint string is forced onto one side of the split, so the validation fold only ever sees complaint text the model never trained on. The gap between the `StratifiedKFold` mean QWK and the `GroupKFold` mean QWK is reported directly as the **shortcut gap** — and it is what justifies treating the tabular-only model (Section 7, Model 4) rather than any text-leveraged model as the honest clinical benchmark for the rest of the notebook.


In [ ]:
text_df = train.merge(cc[['patient_id','chief_complaint_raw']], on='patient_id', how='left')
text_df['chief_complaint_raw'] = text_df['chief_complaint_raw'].fillna('unknown')
y_audit = text_df['triage_acuity'].values - 1

tfidf_a = TfidfVectorizer(max_features=500, ngram_range=(1,2), sublinear_tf=True)
X_text  = tfidf_a.fit_transform(text_df['chief_complaint_raw']).toarray()
lgb_a   = lgb.LGBMClassifier(objective='multiclass', num_class=5, n_estimators=200,
                               learning_rate=0.1, class_weight='balanced',
                               verbose=-1, random_state=SEED, n_jobs=-1)

print('=== STRATIFIED K-FOLD (standard — allows same complaint in train and val) ===')
skf_scores = []
for fold, (tr_i, val_i) in enumerate(
        StratifiedKFold(5, shuffle=True, random_state=SEED).split(X_text, y_audit)):
    lgb_a.fit(X_text[tr_i], y_audit[tr_i])
    q = cohen_kappa_score(y_audit[val_i], lgb_a.predict(X_text[val_i]), weights='quadratic')
    skf_scores.append(q)
    print(f'  Fold {fold+1}: QWK = {q:.4f}')
print(f'  MEAN QWK: {np.mean(skf_scores):.4f}  ← inflated by text memorisation')

print('\n=== GROUP K-FOLD (honest — each complaint string appears in ONE fold only) ===')
groups = text_df['chief_complaint_raw'].values
gkf_scores = []
for fold, (tr_i, val_i) in enumerate(GroupKFold(5).split(X_text, y_audit, groups=groups)):
    lgb_a.fit(X_text[tr_i], y_audit[tr_i])
    q = cohen_kappa_score(y_audit[val_i], lgb_a.predict(X_text[val_i]), weights='quadratic')
    gkf_scores.append(q)
    print(f'  Fold {fold+1}: QWK = {q:.4f}')
print(f'  MEAN QWK: {np.mean(gkf_scores):.4f}  ← true generalisation to unseen complaints')

gap = np.mean(skf_scores) - np.mean(gkf_scores)
print(f'\n  SHORTCUT GAP: {gap:.4f} QWK points = text memorisation artifact')
print('  CONCLUSION: We declare the structured vitals+history model as the honest clinical benchmark.')

In [ ]:
fig = go.Figure()
fig.add_trace(go.Bar(name='StratifiedKFold (leaks text)',
                     x=[f'Fold {i+1}' for i in range(5)], y=skf_scores,
                     marker_color='#d32f2f', text=[f'{s:.3f}' for s in skf_scores],
                     textposition='outside'))
fig.add_trace(go.Bar(name='GroupKFold (honest)',
                     x=[f'Fold {i+1}' for i in range(5)], y=gkf_scores,
                     marker_color='#388e3c', text=[f'{s:.3f}' for s in gkf_scores],
                     textposition='outside'))
fig.update_layout(
    title=f'Shortcut Audit: StratifiedKFold vs GroupKFold QWK<br>'
          f'<sub>Gap = {gap:.3f} QWK points = memorisation, not clinical reasoning</sub>',
    yaxis=dict(title='Quadratic Weighted Kappa', range=[0, 1.15]),
    barmode='group', height=500)
save_fig(fig, 'shortcut_audit.png')

---
## Section 5 — Feature Engineering: 28 Advanced Features

Three feature families are built, each grounded in a real triage instrument or NaN-safe clinical rule rather than generic statistics:

1. **Validated risk scores** — `qsofa` (≥0 points for RR≥22, SBP≤100, altered mental status), an SIRS-style count (`sirs_score`, abnormal temperature/HR/RR), and a Modified Early Warning Score (`mews_score`) computed from piecewise RR/HR/SBP/temperature bands exactly as the bedside MEWS chart defines them. `mental_status_encoded` maps free-text status (unresponsive/drowsy/confused/agitated/alert) onto an ordinal 0–4 scale.
2. **Rule-based safety flags** (`safety_flags`) — five NaN-safe, hard-coded triggers mirroring real red-flag criteria: `flag_stemi` (age≥45, HR>120, SBP<85), `flag_sepsis` (temp>38.8°C or <35.5°C, HR>105, RR>24), `flag_stroke` (GCS<11, age>60), `flag_resp_failure` (SpO2<85, RR>30), and `flag_shock` (shock index>1.4). Missing vitals are filled with normal physiological defaults first so an *absent* reading can never itself trigger a flag — only a genuinely abnormal measured value can.
3. **Missingness-as-signal and imputation** — explicit `*_missing` flags for each vital plus a `vitals_missing_count`, since *why* a value is missing is itself informative (Section 3.3 shows it's MNAR); the actual values are then imputed by group-wise median (grouped on `age_group`/`shift`) so imputation doesn't erase demographic differences. A handful of interaction terms (`age_x_shock`, `gcs_x_spo2`, `news2_x_comorb`) and cyclical encodings of arrival hour (`sin`/`cos`) round out the set, alongside abbreviation-expanded, keyword-tiered chief-complaint features (`kw_critical_life`, `kw_time_sensitive`, …, `kw_severity_score`) built from the regex tiers defined just above.

A fourth layer — Bayesian target-encoding of the complaint text, word- and character-level TF-IDF (100 dims each), and frequency-encoding of categorical columns — is built inside `prepare_fold` in the next cell rather than here, specifically because those encoders must be **refit on the training fold only** for every CV split to guarantee zero leakage from validation data into training.


In [ ]:
ABBREV = {
    r'\bsob\b':'shortness of breath', r'\bc/o\b':'complains of',
    r'\bh/o\b':'history of',          r'\bams\b':'altered mental status',
    r'\bcp\b': 'chest pain',          r'\bdoe\b':'dyspnea on exertion',
    r'\bn/v\b':'nausea vomiting',     r'\bha\b': 'headache',
    r'\bhtn\b':'hypertension',        r'\bdm\b': 'diabetes mellitus',
    r'\bpe\b': 'pulmonary embolism',  r'\bcva\b':'stroke',
    r'\bmi\b': 'myocardial infarction', r'\buti\b':'urinary tract infection',
    r'\bloc\b':'loss of consciousness', r'\bgcs\b':'glasgow coma scale',
    r'\bwob\b':'work of breathing',
}
PAT = {
    'critical':  r'shock|arrest|unconscious|unresponsive|cpr|resuscitat|apnea|pulseless|asystole',
    'time_sens': r'stemi|sepsis|anaphylaxis|pulmonary embolism|aortic|meningitis|eclampsia|stroke|tpa',
    'high':      r'acute|severe|worst|sudden|chest pain|shortness of breath|difficulty breathing|seizure|overdose|trauma|bleeding|hemorrhage',
    'moderate':  r'moderate|pain|fever|vomiting|diarrhea|headache|dizziness|weakness|nausea|swelling',
    'low':       r'mild|minor|routine|follow.?up|prescription|refill|cold|cough|sore throat',
}
def expand_abbrev(text):
    if not isinstance(text, str): return 'unknown'
    t = text.lower().strip()
    for pat, rep in ABBREV.items():
        t = re.sub(pat, rep, t)
    return t
def kw_features(text):
    t  = str(text).lower()
    kc = int(bool(re.search(PAT['critical'],  t)))
    kt = int(bool(re.search(PAT['time_sens'], t)))
    kh = int(bool(re.search(PAT['high'],      t)))
    km = int(bool(re.search(PAT['moderate'],  t)))
    kl = int(bool(re.search(PAT['low'],       t)))
    return {'kw_critical_life':kc,'kw_time_sensitive':kt,'kw_high_acuity':kh,
            'kw_moderate':km,'kw_low':kl,
            'kw_severity_score':3*kc+3*kt+2*kh+km-2*kl,
            'chief_complaint_len':len(str(text))}
print(f'Abbreviation expander: {len(ABBREV)} patterns | Keyword tiers: {len(PAT)}')

In [ ]:
def clinical_thresholds(df):
    df = df.copy()
    ms_map = {'unresponsive':0,'drowsy':1,'confused':1,'agitated':2,'alert':4}
    if 'mental_status_triage' in df.columns:
        df['mental_status_encoded'] = (df['mental_status_triage'].str.lower()
                                        .map(ms_map).fillna(3).astype(int))
        df['mental_status_unres']   = (df['mental_status_triage'].str.lower()=='unresponsive').astype(int)
    df['pain_not_recorded'] = (df['pain_score']==-1).astype(int)
    df['pain_score_clean']  = df['pain_score'].replace(-1, np.nan)
    df['pain_severe']       = (df['pain_score_clean']>=8).fillna(0).astype(int)
    if 'gcs_total'   in df.columns:
        df['gcs_severe']      = (df['gcs_total']<9).fillna(0).astype(int)
        df['gcs_moderate']    = ((df['gcs_total']>=9)&(df['gcs_total']<13)).fillna(0).astype(int)
    if 'spo2'        in df.columns:
        df['spo2_critical']   = (df['spo2']<90).fillna(0).astype(int)
        df['spo2_concerning'] = ((df['spo2']>=90)&(df['spo2']<94)).fillna(0).astype(int)
    if 'systolic_bp' in df.columns:
        df['sbp_hypotensive']  = (df['systolic_bp']<90).fillna(0).astype(int)
        df['sbp_hypertensive'] = (df['systolic_bp']>180).fillna(0).astype(int)
    return df

def clinical_composites(df):
    df = df.copy()
    hx = [c for c in df.columns if c.startswith('hx_')]
    q = np.zeros(len(df))
    if 'respiratory_rate'     in df.columns: q += (df['respiratory_rate']>=22).fillna(0)
    if 'systolic_bp'          in df.columns: q += (df['systolic_bp']<=100).fillna(0)
    if 'mental_status_encoded'in df.columns: q += (df['mental_status_encoded']<=1).fillna(0)
    df['qsofa'] = q.astype(int)
    s = np.zeros(len(df))
    if 'temperature_c'    in df.columns: s += ((df['temperature_c']>38)|(df['temperature_c']<36)).fillna(0)
    if 'heart_rate'       in df.columns: s += (df['heart_rate']>90).fillna(0)
    if 'respiratory_rate' in df.columns: s += (df['respiratory_rate']>20).fillna(0)
    df['sirs_score']    = s.astype(int)
    df['sirs_positive'] = (s>=2).astype(int)
    if hx:
        df['comorbidity_count'] = df[hx].fillna(0).sum(axis=1)
        df['high_comorbidity']  = (df['comorbidity_count']>=5).astype(int)
        cardiac = [c for c in hx if any(k in c for k in ['hypertension','heart_failure','diabetes','cardiac'])]
        resp    = [c for c in hx if any(k in c for k in ['copd','asthma','pulmonary'])]
        if cardiac: df['cardiac_risk_score']     = df[cardiac].fillna(0).sum(axis=1)
        if resp:    df['respiratory_risk_score'] = df[resp].fillna(0).sum(axis=1)

    # MEWS (Modified Early Warning Score)
    mews = np.zeros(len(df))
    if 'respiratory_rate' in df.columns:
        rr_ = df['respiratory_rate'].fillna(16)
        mews += np.where(rr_<9, 2, np.where(rr_<=14, 0, np.where(rr_<=20, 0, np.where(rr_<=29, 2, 3))))
    if 'heart_rate' in df.columns:
        hr_ = df['heart_rate'].fillna(75)
        mews += np.where(hr_<40, 2, np.where(hr_<=50, 1, np.where(hr_<=100, 0, np.where(hr_<=110, 1, np.where(hr_<=129, 2, 3)))))
    if 'systolic_bp' in df.columns:
        sbp_ = df['systolic_bp'].fillna(120)
        mews += np.where(sbp_<70, 3, np.where(sbp_<=80, 2, np.where(sbp_<=100, 1, np.where(sbp_<=199, 0, 2))))
    if 'temperature_c' in df.columns:
        tmp_ = df['temperature_c'].fillna(37)
        mews += np.where(tmp_<35, 2, np.where(tmp_<38.5, 0, 2))
    df['mews_score'] = mews.astype(int)
    df['mews_high']  = (mews >= 5).astype(int)
    return df

def safety_flags(df):
    """NaN-safe: fill with normal physiological values so missing data never triggers a flag."""
    df  = df.copy()
    hr  = df['heart_rate'].fillna(75)       if 'heart_rate'       in df.columns else pd.Series(75,  index=df.index)
    sbp = df['systolic_bp'].fillna(120)     if 'systolic_bp'      in df.columns else pd.Series(120, index=df.index)
    rr  = df['respiratory_rate'].fillna(16) if 'respiratory_rate' in df.columns else pd.Series(16,  index=df.index)
    spo = df['spo2'].fillna(98)             if 'spo2'             in df.columns else pd.Series(98,  index=df.index)
    tmp = df['temperature_c'].fillna(37)    if 'temperature_c'    in df.columns else pd.Series(37,  index=df.index)
    gcs = df['gcs_total'].fillna(15)        if 'gcs_total'        in df.columns else pd.Series(15,  index=df.index)
    age = df['age'].fillna(40)              if 'age'              in df.columns else pd.Series(40,  index=df.index)
    si  = df['shock_index'].fillna(0.6)     if 'shock_index'      in df.columns else pd.Series(0.6, index=df.index)
    df['flag_stemi']        = ((age>=45)&(hr>120)&(sbp<85)).astype(int)
    df['flag_sepsis']       = (((tmp>38.8)|(tmp<35.5))&(hr>105)&(rr>24)).astype(int)
    df['flag_stroke']       = ((gcs<11)&(age>60)).astype(int)
    df['flag_resp_failure'] = ((spo<85)&(rr>30)).astype(int)
    df['flag_shock']        = (si>1.4).astype(int)
    df['any_safety_flag']   = (df[['flag_stemi','flag_sepsis','flag_stroke',
                                    'flag_resp_failure','flag_shock']].sum(axis=1)>0).astype(int)
    return df

print('Clinical feature functions defined.')

In [ ]:
def build_master_df(df, cc_df, hist_df):
    df = df.copy().reset_index(drop=True)
    df = df.merge(cc_df[['patient_id','chief_complaint_raw']], on='patient_id', how='left')
    df = df.merge(hist_df, on='patient_id', how='left')
    df['cc_expanded'] = df['chief_complaint_raw'].apply(expand_abbrev)
    kw = pd.DataFrame(list(df['cc_expanded'].apply(kw_features))).reset_index(drop=True)
    df = pd.concat([df.reset_index(drop=True), kw], axis=1)
    df = clinical_thresholds(df)
    df = clinical_composites(df)
    df = safety_flags(df)
    miss_c = [c for c in ['systolic_bp','diastolic_bp','mean_arterial_pressure','pulse_pressure',
                           'shock_index','respiratory_rate','temperature_c','spo2'] if c in df.columns]
    for c in miss_c:
        df[f'{c}_missing'] = df[c].isnull().astype(int)
    mc = [f'{c}_missing' for c in miss_c if f'{c}_missing' in df.columns]
    df['vitals_missing_count'] = df[mc].sum(axis=1) if mc else 0
    num_imp = [c for c in ['systolic_bp','diastolic_bp','mean_arterial_pressure','pulse_pressure',
                            'shock_index','respiratory_rate','temperature_c','spo2','heart_rate',
                            'gcs_total','bmi','pain_score_clean'] if c in df.columns]
    grp = ['age_group','shift'] if 'age_group' in df.columns and 'shift' in df.columns else []
    for c in num_imp:
        if grp:
            df[c] = df.groupby(grp)[c].transform(lambda x: x.fillna(x.median()))
        df[c] = df[c].fillna(df[c].median())
    if 'arrival_hour' in df.columns:
        df['arrival_hour_sin'] = np.sin(2*np.pi*df['arrival_hour']/24)
        df['arrival_hour_cos'] = np.cos(2*np.pi*df['arrival_hour']/24)
    if 'age' in df.columns:
        df['elderly']      = (df['age']>=65).astype(int)
        df['pediatric']    = (df['age']<16).astype(int)
        df['very_elderly'] = (df['age']>=80).astype(int)
        if 'shock_index' in df.columns:
            df['age_x_shock'] = df['age'] * df['shock_index'].fillna(0)
    if 'gcs_total' in df.columns and 'spo2' in df.columns:
        df['gcs_x_spo2'] = df['gcs_total'] * df['spo2'].fillna(95)
    if 'news2_score' in df.columns and 'comorbidity_count' in df.columns:
        df['news2_x_comorb'] = df['news2_score'].fillna(0) * df['comorbidity_count'].fillna(0)
    if 'arrival_mode' in df.columns:
        df['high_risk_arrival'] = df['arrival_mode'].isin(['ambulance','helicopter']).astype(int)
    if 'arrival_day' in df.columns:
        df['weekend'] = df['arrival_day'].isin(['Saturday','Sunday']).astype(int)
    if 'shift' in df.columns:
        df['night_shift'] = (df['shift']=='night').astype(int)
    return df.loc[:, ~df.columns.duplicated()]

print('Building master dataframes...')
train_master = build_master_df(train, cc, history)
test_master  = build_master_df(test,  cc, history)
print(f'train_master: {train_master.shape} | test_master: {test_master.shape}')
new_feats = [c for c in train_master.columns
             if c not in list(train.columns)+list(history.columns)]
print(f'New engineered features: {len(new_feats)}')

In [ ]:
NLP_PCA_COLS = [f'nlp_pca_{i}' for i in range(32)] + ['nlp_high_risk_prob']
CAT_COLS_ALL = ['arrival_mode','arrival_day','sex','age_group','shift','arrival_season',
                'transport_origin','chief_complaint_system','insurance_type','language',
                'mental_status_triage']
TFIDF_COLS   = [f'tfw_{i}' for i in range(100)] + [f'tfc_{i}' for i in range(100)]

def prepare_fold(tr_df, val_df, te_df, include_nlp=True):
    """All encoders fitted on tr_df only — zero leakage guaranteed."""
    tr_df  = tr_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    te_df  = te_df.reset_index(drop=True)
    gm  = tr_df['triage_acuity'].mean()
    st  = tr_df.groupby('cc_expanded')['triage_acuity'].agg(['mean','count'])
    st['enc'] = (st['count']*st['mean']+10*gm)/(st['count']+10)
    em  = st['enc'].to_dict()
    tr_cc  = tr_df['cc_expanded'].map(em).fillna(gm).values
    val_cc = val_df['cc_expanded'].map(em).fillna(gm).values
    te_cc  = te_df['cc_expanded'].map(em).fillna(gm).values
    tw = TfidfVectorizer(max_features=100, ngram_range=(1,2), sublinear_tf=True)
    tc = TfidfVectorizer(max_features=100, ngram_range=(2,4), analyzer='char_wb', sublinear_tf=True)
    t_tr  = tr_df['cc_expanded'].fillna('unknown')
    t_val = val_df['cc_expanded'].fillna('unknown')
    t_te  = te_df['cc_expanded'].fillna('unknown')
    tw.fit(t_tr); tc.fit(t_tr)
    def tfidf(texts):
        return np.hstack([tw.transform(texts).toarray(), tc.transform(texts).toarray()])
    tfidf_tr  = tfidf(t_tr)
    tfidf_val = tfidf(t_val)
    tfidf_te  = tfidf(t_te)
    cat_cols = [c for c in CAT_COLS_ALL if c in tr_df.columns]
    freq_tr, freq_val, freq_te, freq_names = [], [], [], []
    for col in cat_cols:
        fm = tr_df[col].value_counts(normalize=True).to_dict()
        freq_tr.append(tr_df[col].map(fm).fillna(0).values)
        freq_val.append(val_df[col].map(fm).fillna(0).values)
        freq_te.append(te_df[col].map(fm).fillna(0).values)
        freq_names.append(f'{col}_freq')
    EXCLUDE = set(['patient_id','triage_acuity','disposition','ed_los_hours',
                   'chief_complaint_raw','cc_expanded']+cat_cols+NLP_PCA_COLS+TFIDF_COLS+freq_names+['cc_bayesian_te'])
    num_cols = [c for c in tr_df.columns if c not in EXCLUDE and tr_df[c].dtype not in ['object','O']]
    def build(df, cc_enc, tfidf_arr, freq_arrs, use_nlp):
        parts  = [df[num_cols].fillna(0).values, cc_enc.reshape(-1,1), tfidf_arr]
        cnames = num_cols + ['cc_bayesian_te'] + TFIDF_COLS
        if freq_arrs:
            parts.append(np.column_stack(freq_arrs))
            cnames += freq_names
        if use_nlp:
            nlp_p = [c for c in NLP_PCA_COLS if c in df.columns]
            if nlp_p:
                parts.append(df[nlp_p].fillna(0).values)
                cnames += nlp_p
        X = np.hstack(parts)
        assert X.shape[1]==len(cnames), f'Column mismatch: {X.shape[1]} vs {len(cnames)}'
        res = pd.DataFrame(X, columns=cnames)
        return res.loc[:, ~res.columns.duplicated()]
    X_tr  = build(tr_df,  tr_cc,  tfidf_tr,  freq_tr,  include_nlp)
    X_val = build(val_df, val_cc, tfidf_val, freq_val, include_nlp)
    X_te  = build(te_df,  te_cc,  tfidf_te,  freq_te,  include_nlp)
    return X_tr, X_val, X_te

train_master = train_master.loc[:,~train_master.columns.duplicated()].reset_index(drop=True)
test_master  = test_master.loc[:,~test_master.columns.duplicated()].reset_index(drop=True)
print(f'prepare_fold defined. train_master: {train_master.shape}')

---
## Section 6 — Semantic NLP: sentence-transformers + UMAP

Section 4 shows that raw TF-IDF on chief-complaint text is close to a label lookup table in this synthetic dataset — a shortcut, not clinical reasoning. To extract NLP signal that generalizes rather than memorizes, every complaint (train + test combined, encoded once) is passed through `all-MiniLM-L6-v2` sentence embeddings, which capture clinical *meaning* (synonymy, negation, severity language) rather than exact n-gram identity. The 384-dim embeddings are compressed via PCA into 32 components (`nlp_pca_0…31`) for the tabular models, and a separate logistic-regression classifier (`lr_nlp`) is trained directly on the raw embeddings to predict "is this patient ESI-1/2" — its output probability becomes the single feature `nlp_high_risk_prob`. A 2-D UMAP projection of an 8K-patient sample is plotted purely as a sanity check: patients with semantically similar complaints should cluster together by ESI regardless of exact wording.


In [ ]:
print('Loading all-MiniLM-L6-v2...')
st_model = SentenceTransformer('all-MiniLM-L6-v2', device=DEVICE)

all_cc = pd.concat([train_master[['patient_id','cc_expanded']],
                    test_master[['patient_id','cc_expanded']]]).reset_index(drop=True)
print(f'Encoding {len(all_cc):,} complaints on {DEVICE}...')
all_emb = st_model.encode(all_cc['cc_expanded'].fillna('unknown').tolist(),
                           batch_size=512, show_progress_bar=True, device=DEVICE)
print(f'Embeddings: {all_emb.shape}')

n_train   = len(train_master)
train_emb = all_emb[:n_train]
test_emb  = all_emb[n_train:]

pca    = PCA(n_components=32, random_state=SEED)
tr_pca = pca.fit_transform(train_emb)
te_pca = pca.transform(test_emb)
print(f'PCA variance explained (32 components): {pca.explained_variance_ratio_.sum()*100:.1f}%')

for mdf in [train_master, test_master]:
    to_drop = [c for c in NLP_PCA_COLS if c in mdf.columns]
    if to_drop:
        mdf.drop(columns=to_drop, inplace=True)

for i in range(32):
    train_master[f'nlp_pca_{i}'] = tr_pca[:, i]
    test_master[f'nlp_pca_{i}']  = te_pca[:, i]

y_bin  = (train_master['triage_acuity']<=2).astype(int).values
lr_nlp = LogisticRegression(max_iter=500, class_weight='balanced', random_state=SEED)
lr_nlp.fit(train_emb, y_bin)
train_master['nlp_high_risk_prob'] = lr_nlp.predict_proba(train_emb)[:,1]
test_master['nlp_high_risk_prob']  = lr_nlp.predict_proba(test_emb)[:,1]

train_master = train_master.loc[:,~train_master.columns.duplicated()].reset_index(drop=True)
test_master  = test_master.loc[:,~test_master.columns.duplicated()].reset_index(drop=True)
print(f'train_master: {train_master.shape} | test_master: {test_master.shape}')

In [ ]:
print('Running UMAP (8K sample)...')
sidx   = np.random.choice(n_train, size=min(8000,n_train), replace=False)
red    = umap.UMAP(n_components=2, random_state=SEED, n_neighbors=15, min_dist=0.1)
umap2d = red.fit_transform(train_emb[sidx])
udf = pd.DataFrame({'x':umap2d[:,0],'y':umap2d[:,1],
                    'ESI':train_master.iloc[sidx]['triage_acuity'].astype(str).values,
                    'complaint':train_master.iloc[sidx]['chief_complaint_raw'].fillna('?').values})
fig = px.scatter(udf, x='x', y='y', color='ESI', hover_data=['complaint'],
                 color_discrete_map={'1':'#d32f2f','2':'#f57c00','3':'#fbc02d','4':'#388e3c','5':'#1976d2'},
                 title='UMAP — Chief Complaint Semantic Embeddings<br>'
                       '<sub>Hover to see raw complaint text. ESI-1/2 form distinct clusters. '
                       'Unlike TF-IDF, semantic embeddings generalise to paraphrased complaints.</sub>',
                 opacity=0.6)
fig.update_traces(marker=dict(size=3))
fig.update_layout(height=580)
save_fig(fig, 'umap_embeddings.png')

---
## Section 7 — Model Training: 4-Model GPU Stacked Ensemble

XGBoost's hyperparameters (depth, learning rate, subsampling, regularization) are tuned first with 30 Optuna trials under 3-fold CV, optimizing mean QWK directly (`xgb_objective`). The resulting `BEST_XGB_PARAMS` then trains inside the main 5-fold `StratifiedKFold` loop alongside three other learners, producing leakage-free out-of-fold (OOF) predictions for all four:
- **Model 0 — XGBoost (GPU)** with the Optuna-tuned params, full feature set including NLP.
- **Model 1 — LightGBM (CPU)** with manually-set params, full feature set including NLP.
- **Model 2 — CatBoost (GPU)** with explicit `class_weights` derived from inverse class frequency (clipped to 1–8×) so the rare ESI-1 class isn't drowned out during training — not just down-weighted at evaluation time.
- **Model 3 — XGBoost (tabular-only)**, built with `include_nlp=False` — this is the **honest clinical benchmark** flagged in Section 4, since it cannot exploit the text-memorization shortcut.

A `LogisticRegression` meta-model stacks the four models' 5-class OOF probabilities into a final prediction (`stack_oof`). The remaining cells in this section add: 1,000-sample bootstrap 95% confidence intervals on each model's QWK (to check whether differences between models are statistically meaningful or just noise), a row-normalized confusion matrix with undertriage cells (predicted-less-severe-than-true) outlined in red, a full per-class precision/recall/F1 report with ESI-1 and ESI-2 recall reported separately as the clinical safety metrics, and per-class calibration curves with Brier scores to check whether predicted probabilities are trustworthy, not just the predicted class.


In [ ]:
print('Optuna HPO — XGBoost GPU (30 trials)...')

def xgb_objective(trial):
    params = {
        'objective':'multi:softprob','num_class':5,'eval_metric':'mlogloss',
        'tree_method':'hist','device':XGB_DEVICE,'random_state':SEED,
        'verbosity':0,'early_stopping_rounds':30,
        'n_estimators':     trial.suggest_int('n_estimators',300,1000),
        'learning_rate':    trial.suggest_float('learning_rate',0.01,0.15,log=True),
        'max_depth':        trial.suggest_int('max_depth',4,8),
        'subsample':        trial.suggest_float('subsample',0.6,1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree',0.6,1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha',0.0,1.0),
        'reg_lambda':       trial.suggest_float('reg_lambda',0.0,1.0),
        'min_child_weight': trial.suggest_int('min_child_weight',1,10),
    }
    tm_ = train_master.reset_index(drop=True)
    y_  = tm_['triage_acuity'].values - 1
    scores = []
    for tr_i,val_i in StratifiedKFold(3,shuffle=True,random_state=SEED).split(tm_,y_):
        tr_  = tm_.iloc[tr_i].reset_index(drop=True)
        val_ = tm_.iloc[val_i].reset_index(drop=True)
        X_tr_,X_val_,_ = prepare_fold(tr_,val_,test_master)
        sw = compute_sample_weight('balanced',y_[tr_i])
        m  = xgb.XGBClassifier(**params)
        m.fit(X_tr_,y_[tr_i],sample_weight=sw,eval_set=[(X_val_,y_[val_i])],verbose=False)
        scores.append(cohen_kappa_score(y_[val_i],m.predict(X_val_),weights='quadratic'))
    return np.mean(scores)

study = optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(xgb_objective,n_trials=30,show_progress_bar=True)
BEST_XGB_PARAMS = study.best_params.copy()
BEST_XGB_PARAMS.update({'objective':'multi:softprob','num_class':5,'eval_metric':'mlogloss',
                          'tree_method':'hist','device':XGB_DEVICE,'random_state':SEED,
                          'verbosity':0,'early_stopping_rounds':50})
print(f'Best QWK: {study.best_value:.4f}')

In [ ]:
y_train = train_master['triage_acuity'].values - 1
tm      = train_master.reset_index(drop=True)
skf     = StratifiedKFold(n_splits=N_FOLDS,shuffle=True,random_state=SEED)

N_MODELS     = 4
oof_preds    = np.zeros((len(tm),5,N_MODELS))
test_preds   = np.zeros((len(test_master),5,N_MODELS))
qwk_scores   = {i:[] for i in range(N_MODELS)}
models_store = {i:[] for i in range(N_MODELS)}

for fold,(tr_idx,val_idx) in enumerate(skf.split(tm,y_train)):
    print(f'\n=== FOLD {fold+1}/{N_FOLDS} ===')
    tr_df  = tm.iloc[tr_idx].reset_index(drop=True)
    val_df = tm.iloc[val_idx].reset_index(drop=True)
    y_tr,y_val = y_train[tr_idx],y_train[val_idx]
    sw_tr = compute_sample_weight('balanced',y_tr)
    X_tr,X_val,X_te = prepare_fold(tr_df,val_df,test_master)

    m0 = xgb.XGBClassifier(**BEST_XGB_PARAMS)
    m0.fit(X_tr,y_tr,sample_weight=sw_tr,eval_set=[(X_val,y_val)],verbose=False)
    p0v=m0.predict_proba(X_val); p0t=m0.predict_proba(X_te)
    oof_preds[val_idx,:,0]=p0v; test_preds[:,:,0]+=p0t/N_FOLDS; models_store[0].append(m0)
    q0=cohen_kappa_score(y_val,np.argmax(p0v,1),weights='quadratic'); qwk_scores[0].append(q0)
    print(f'  XGBoost GPU:       QWK={q0:.4f}')

    m1 = lgb.LGBMClassifier(objective='multiclass',num_class=5,n_estimators=600,
                              learning_rate=0.05,num_leaves=63,min_child_samples=20,
                              subsample=0.8,colsample_bytree=0.8,class_weight='balanced',
                              device='cpu',n_jobs=-1,random_state=SEED,verbose=-1)
    m1.fit(X_tr,y_tr,eval_set=[(X_val,y_val)],
           callbacks=[lgb.early_stopping(50,verbose=False),lgb.log_evaluation(-1)])
    p1v=m1.predict_proba(X_val); p1t=m1.predict_proba(X_te)
    oof_preds[val_idx,:,1]=p1v; test_preds[:,:,1]+=p1t/N_FOLDS; models_store[1].append(m1)
    q1=cohen_kappa_score(y_val,np.argmax(p1v,1),weights='quadratic'); qwk_scores[1].append(q1)
    print(f'  LightGBM CPU:      QWK={q1:.4f}')

    # CatBoost class weights: derived from inverse class frequency
    # ESI-1: ~4% → weight 5 | ESI-2: ~13% → weight 4 | ESI-3-5: majority → weight 1
    # Clinical rationale: underpredicting ESI-1 is catastrophic; weights reflect this asymmetry
    esi_counts_train = np.bincount(y_tr, minlength=5)
    max_count = esi_counts_train.max()
    cat_weights = np.clip(max_count / (esi_counts_train + 1e-8), 1, 8).tolist()
    cat_weights = [round(w, 1) for w in cat_weights]
    m2 = CatBoostClassifier(iterations=600,learning_rate=0.05,depth=6,
                             loss_function='MultiClass',task_type=CAT_DEVICE,
                             class_weights=cat_weights,random_seed=SEED,verbose=0)
    m2.fit(X_tr,y_tr,eval_set=(X_val,y_val),early_stopping_rounds=50)
    p2v=m2.predict_proba(X_val); p2t=m2.predict_proba(X_te)
    oof_preds[val_idx,:,2]=p2v; test_preds[:,:,2]+=p2t/N_FOLDS; models_store[2].append(m2)
    q2=cohen_kappa_score(y_val,np.argmax(p2v,1),weights='quadratic'); qwk_scores[2].append(q2)
    print(f'  CatBoost GPU:      QWK={q2:.4f}')

    X_tr3,X_val3,X_te3 = prepare_fold(tr_df,val_df,test_master,include_nlp=False)
    m3 = xgb.XGBClassifier(objective='multi:softprob',num_class=5,eval_metric='mlogloss',
                             tree_method='hist',device=XGB_DEVICE,n_estimators=500,
                             learning_rate=0.05,max_depth=6,early_stopping_rounds=50,
                             random_state=SEED,verbosity=0)
    m3.fit(X_tr3,y_tr,sample_weight=sw_tr,eval_set=[(X_val3,y_val)],verbose=False)
    p3v=m3.predict_proba(X_val3); p3t=m3.predict_proba(X_te3)
    oof_preds[val_idx,:,3]=p3v; test_preds[:,:,3]+=p3t/N_FOLDS; models_store[3].append(m3)
    q3=cohen_kappa_score(y_val,np.argmax(p3v,1),weights='quadratic'); qwk_scores[3].append(q3)
    print(f'  XGBoost tabular:   QWK={q3:.4f}  <- HONEST BENCHMARK')

meta_train      = oof_preds.reshape(len(tm),-1)
meta_test_arr   = test_preds.reshape(len(test_master),-1)
meta_model      = LogisticRegression(max_iter=1000,C=1.0,random_state=SEED)
meta_model.fit(meta_train,y_train)
stack_oof       = meta_model.predict(meta_train)
stack_test_prob = meta_model.predict_proba(meta_test_arr)
stack_test_pred = meta_model.predict(meta_test_arr)
stack_qwk       = cohen_kappa_score(y_train,stack_oof,weights='quadratic')
oof_prob_stack  = meta_model.predict_proba(meta_train)

MODEL_NAMES = ['XGBoost+NLP (GPU)','LightGBM+NLP (CPU)','CatBoost (GPU)','XGBoost tabular-only']
print('\n=== RESULTS ===')
for i,n in enumerate(MODEL_NAMES):
    s=qwk_scores[i]; print(f'  {n:<35} QWK: {np.mean(s):.4f} ± {np.std(s):.4f}')
print(f'  {"Stacked Ensemble":<35} QWK: {stack_qwk:.4f}')

In [ ]:
# ── Bootstrap QWK confidence intervals ───────────────────────────────────────
print('=== BOOTSTRAP QWK 95% CONFIDENCE INTERVALS ===')
print('(1000 bootstrap samples on OOF predictions)')
np.random.seed(SEED)
n_boot = 1000
boot_qwks = {i: [] for i in range(N_MODELS)}
boot_stack = []

for _ in range(n_boot):
    idx = np.random.choice(len(y_train), size=len(y_train), replace=True)
    for i in range(N_MODELS):
        preds_i = np.argmax(oof_preds[idx, :, i], axis=1)
        try:
            boot_qwks[i].append(cohen_kappa_score(y_train[idx], preds_i, weights='quadratic'))
        except Exception:
            boot_qwks[i].append(np.nan)
    try:
        boot_stack.append(cohen_kappa_score(y_train[idx], stack_oof[idx], weights='quadratic'))
    except Exception:
        boot_stack.append(np.nan)

print(f'{"Model":<38} {"Mean QWK":<12} {"95% CI"}')
print('-' * 65)
for i, name in enumerate(MODEL_NAMES):
    vals = [v for v in boot_qwks[i] if not np.isnan(v)]
    lo, hi = np.percentile(vals, [2.5, 97.5])
    print(f'{name:<38} {np.mean(vals):<12.4f} [{lo:.4f}, {hi:.4f}]')

stack_vals = [v for v in boot_stack if not np.isnan(v)]
lo, hi = np.percentile(stack_vals, [2.5, 97.5])
print(f'{"Stacked Ensemble":<38} {np.mean(stack_vals):<12.4f} [{lo:.4f}, {hi:.4f}]')
print()
print('Interpretation: overlapping CIs indicate models are statistically equivalent;')
print('non-overlapping CIs indicate genuine performance differences.')

# Plot with error bars
ci_df_rows = []
for i, name in enumerate(MODEL_NAMES):
    vals = [v for v in boot_qwks[i] if not np.isnan(v)]
    ci_df_rows.append({'Model':name,'Mean':np.mean(vals),
                        'Lo':np.percentile(vals,2.5),'Hi':np.percentile(vals,97.5)})
stack_vals2 = [v for v in boot_stack if not np.isnan(v)]
ci_df_rows.append({'Model':'Stacked Ensemble','Mean':np.mean(stack_vals2),
                    'Lo':np.percentile(stack_vals2,2.5),'Hi':np.percentile(stack_vals2,97.5)})
ci_df = pd.DataFrame(ci_df_rows)

fig = go.Figure()
fig.add_trace(go.Bar(x=ci_df['Model'], y=ci_df['Mean'],
                     error_y=dict(type='data', symmetric=False,
                                  array=ci_df['Hi']-ci_df['Mean'],
                                  arrayminus=ci_df['Mean']-ci_df['Lo']),
                     marker_color='#1976d2', text=ci_df['Mean'].round(4),
                     textposition='outside'))
fig.update_layout(title='Model Comparison — QWK with 95% Bootstrap Confidence Intervals',
                  yaxis=dict(title='QWK', range=[0, 1.1]), height=460)
save_fig(fig, 'model_comparison_bootstrap_ci.png')


In [ ]:
# Confusion matrix
cm     = confusion_matrix(y_train, stack_oof)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
fig, ax = plt.subplots(figsize=(8,6))
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues', ax=ax,
            xticklabels=[f'Pred ESI-{i}' for i in range(1,6)],
            yticklabels=[f'True ESI-{i}' for i in range(1,6)])
for i in range(5):
    for j in range(5):
        if j > i:
            ax.add_patch(plt.Rectangle((j,i),1,1,fill=False,edgecolor='red',lw=2))
ax.set_title(f'Confusion Matrix — Stacked Ensemble OOF (QWK={stack_qwk:.4f})\n'
             f'Row-normalised % | Red borders = undertriage (dangerous errors)')
ax.set_ylabel('True ESI'); ax.set_xlabel('Predicted ESI')
plt.tight_layout()
save_mpl('lgbm_confusion_matrix.png')

# Model comparison bar chart
res_df = pd.DataFrame({'Model':MODEL_NAMES+['Stacked Ensemble'],
                        'QWK':[np.mean(qwk_scores[i]) for i in range(N_MODELS)]+[stack_qwk]})
fig = px.bar(res_df,x='Model',y='QWK',text='QWK',color='QWK',
             color_continuous_scale='Greens',title='Model Comparison — OOF QWK Scores')
fig.update_traces(texttemplate='%{text:.4f}',textposition='outside')
fig.update_layout(height=440)
save_fig(fig,'model_comparison_qwk.png')

In [ ]:
# ── Per-class classification report + calibration curves ─────────────────────
from sklearn.metrics import classification_report

print('=== PER-CLASS CLASSIFICATION REPORT — STACKED ENSEMBLE (OOF) ===')
print(classification_report(
    y_train, stack_oof,
    target_names=['ESI-1 (Immediate)','ESI-2 (Emergent)','ESI-3 (Urgent)',
                  'ESI-4 (Less Urgent)','ESI-5 (Non-Urgent)'],
    digits=4))

# Clinical safety metric — ESI-1 recall is the money number
esi1_recall = (stack_oof[y_train==0]==0).mean()
esi2_recall = (stack_oof[y_train==1]==1).mean()
print(f'ESI-1 Recall (Critical patients caught):   {esi1_recall:.4f} ({esi1_recall*100:.1f}%)')
print(f'ESI-2 Recall (Emergent patients caught):   {esi2_recall:.4f} ({esi2_recall*100:.1f}%)')
ut_rate = (stack_oof > y_train).mean()
print(f'Overall undertriage rate:                  {ut_rate:.4f} ({ut_rate*100:.1f}%)')

# ── Calibration curves ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ci, cn in enumerate(['ESI-1','ESI-2','ESI-3','ESI-4','ESI-5']):
    y_bin  = (y_train == ci).astype(int)
    prob_c = oof_prob_stack[:, ci]
    try:
        frac_pos, mean_pred = calibration_curve(y_bin, prob_c, n_bins=10, strategy='quantile')
        bs = brier_score_loss(y_bin, prob_c)
        axes[ci].plot(mean_pred, frac_pos, 's-', color='#1976d2', label=f'Model (BS={bs:.3f})')
        axes[ci].plot([0,1],[0,1],'--',color='gray',label='Perfect calibration')
        axes[ci].set_title(f'{cn}\nBrier={bs:.3f}', fontsize=10)
        axes[ci].set_xlabel('Mean Predicted Prob')
        axes[ci].set_ylabel('Fraction of Positives')
        axes[ci].legend(fontsize=7)
        axes[ci].set_xlim(0,1); axes[ci].set_ylim(0,1)
    except Exception:
        axes[ci].text(0.5, 0.5, 'Insufficient data', ha='center', va='center')
        axes[ci].set_title(cn)
plt.suptitle('Calibration Curves by ESI Class — Reliability Diagrams\n'
             'Points on diagonal = perfectly calibrated', fontsize=12, y=1.02)
plt.tight_layout()
save_mpl('calibration_curves.png')
print('Saved: calibration_curves.png')


---
## Section 8 — Threshold Optimization

Argmax classification ignores the ordinal structure of ESI and can produce a degenerate prediction distribution — for example, the model never predicting ESI-1 because it's only ~4% of cases. Here, four cumulative-probability cutpoints (`OPTIMAL_THRESHOLDS`, one fewer than the number of classes) are optimized via differential evolution directly against QWK, with a penalty term that activates if any class's predicted share falls below 0.5%. This converts the model's raw probabilities into a decision rule a real triage system could deploy, where rare-but-critical classes aren't silently dropped to inflate aggregate accuracy.


In [ ]:
def apply_thresholds(probs, thresholds):
    \"\"\"
    Ordinal threshold classifier.
    thresholds: 4 values t1..t4 in [0,1].
    Checks cumulative probabilities left-to-right (ESI 1 to 4).
    The first threshold exceeded defines the class. If none, assigns ESI-5.
    \"\"\"
    cumprob = np.cumsum(probs, axis=1)
    exceeds = cumprob[:, :4] >= np.array(thresholds)
    # Append a column of True to catch anything that doesn't exceed the first 4 thresholds (becomes ESI-5)
    exceeds = np.column_stack([exceeds, np.ones(len(probs), dtype=bool)])
    # argmax returns the index of the FIRST True value. +1 for 1-indexed ESI.
    return np.argmax(exceeds, axis=1) + 1

def objective(thresholds):
    preds = apply_thresholds(oof_prob_stack, thresholds)
    # y_train is 0-indexed (0..4), preds is 1-indexed (1..5)
    qwk   = cohen_kappa_score(y_train, preds - 1, weights='quadratic')
    dist  = np.bincount(preds, minlength=6)[1:] / len(preds)   # ESI 1..5 shares
    # Penalty: any class below 0.5% predicted share -> large cost
    penalty = sum(10.0 * max(0, 0.005 - d) for d in dist)
    return -qwk + penalty


print('Running Differential Evolution threshold optimization...')
result = differential_evolution(
    objective, bounds=[(0.05, 0.95)] * 4,
    seed=SEED, maxiter=300, tol=1e-6,
    popsize=20, mutation=(0.5, 1.5), recombination=0.7,
    workers=1
)
OPTIMAL_THRESHOLDS = result.x
opt_oof_preds = apply_thresholds(oof_prob_stack, OPTIMAL_THRESHOLDS)
opt_qwk       = cohen_kappa_score(y_train, opt_oof_preds - 1, weights='quadratic')

print(f'Raw ensemble QWK:   {stack_qwk:.4f}')
print(f'Optimized QWK:      {opt_qwk:.4f}  (+{opt_qwk - stack_qwk:.4f})')
print(f'Optimal thresholds: {OPTIMAL_THRESHOLDS.round(4)}')
print('\nOptimized OOF distribution:')
for esi, cnt in pd.Series(opt_oof_preds).value_counts().sort_index().items():
    bar = chr(9608) * int(cnt / 500)
    print(f'  ESI-{esi}: {cnt:>5,} ({cnt/len(opt_oof_preds)*100:.1f}%) {bar}')

# Apply same thresholds to test set — meta_probs shape: (n_test, 5)
meta_probs               = meta_model.predict_proba(meta_test_arr)
test_master['final_esi'] = apply_thresholds(meta_probs, OPTIMAL_THRESHOLDS)
print(f'\nTest predictions: {pd.Series(test_master["final_esi"]).value_counts().sort_index().to_dict()}')

---
## Section 9 — Conformal Prediction

A point prediction with no uncertainty estimate is risky in a clinical setting — a confidently-predicted ESI-3 and a barely-confident ESI-3 should not be handled identically by a busy triage desk. The last 20% of OOF predictions is held out as a calibration slice; its non-conformity scores (`1 − P(true class)`) define a quantile threshold `q` for a target miscoverage rate `alpha`. For each new patient, every class whose `1 − P(class)` falls at or below `q` is included in that patient's *prediction set* — sometimes spanning more than one ESI level — with a guaranteed coverage rate (e.g. 90% at α=0.10). Test patients whose set spans multiple acuity levels are explicitly flagged in `test_master['needs_review']`, which is exactly the population that should be escalated for manual clinician review rather than auto-triaged.


In [ ]:
cal_size = int(0.2*len(y_train))
cal_probs = oof_prob_stack[-cal_size:]
cal_true  = y_train[-cal_size:]
nonconf   = 1 - cal_probs[np.arange(cal_size), cal_true]

def conformal_sets(probs, alpha=0.10):
    q    = np.quantile(nonconf, 1-alpha)
    sets = []
    for p in probs:
        s = [c for c in range(5) if (1-p[c])<=q]
        sets.append(s if s else [int(np.argmax(p))])
    return sets

print(f'{"Alpha":<8}{"Coverage":<14}{"Avg Set Size":<16}{"Single %":<14}{"Flag Rate"}')
for alpha in [0.05,0.10,0.20]:
    sets     = conformal_sets(cal_probs,alpha)
    coverage = np.mean([cal_true[i] in s for i,s in enumerate(sets)])
    avg_sz   = np.mean([len(s) for s in sets])
    single   = np.mean([len(s)==1 for s in sets])*100
    flag     = np.mean([len(s)>1 for s in sets])*100
    print(f'{alpha:<8.2f}{coverage:<14.3f}{avg_sz:<16.2f}{single:<14.1f}{flag:.1f}%')

test_conf_sets               = conformal_sets(stack_test_prob,alpha=0.10)
test_master['conformal_set'] = [str(sorted([x+1 for x in s])) for s in test_conf_sets]
test_master['needs_review']  = [len(s)>1 for s in test_conf_sets]
n_rev = test_master['needs_review'].sum()
print(f'\nTest patients flagged for senior review (α=0.10): {n_rev:,} ({n_rev/len(test_master)*100:.1f}%)')

---
## Section 10 — Explainability Suite

A model a clinician cannot interrogate is a model a clinician should not trust at the bedside. This section layers four explanation methods on an 80/20 train/validation split of the LightGBM model (Model 1), each answering a different question:
- **SHAP** — `TreeExplainer` on a 3,000-patient sample produces global beeswarm/bar summaries per ESI class plus per-patient waterfall plots for three illustrative cases (a correctly-identified ESI-1, a dangerous ESI-2→3 undertriage miss, and an ESI-3 edge case).
- **LIME** — an independent local linear approximation around two specific patients (an ESI-1 critical case and an ESI-3 borderline case), as a cross-check using a different explanation method than SHAP.
- **Clinical rationale generator** — a hand-built `INTERP` lookup table translates raw feature values (e.g. `news2_score=6`) into the same risk-tier language ("high risk") a triage nurse already uses, then prints the top-3 SHAP-ranked drivers for example patients in plain language.
- **SHAP-based counterfactual analysis** — reads the SHAP values for the ESI-2 class on one ESI-3 patient and reports which features have the most positive/negative attribution toward ESI-2, as a fast proxy for "what would push this patient to a more urgent class." This is an attribution-based approximation rather than a true optimization-based counterfactual search (e.g. `dice_ml`), which is a reasonable trade-off for this notebook scope.


In [ ]:
tr0  = tm.iloc[:int(0.8*len(tm))].reset_index(drop=True)
val0 = tm.iloc[int(0.8*len(tm)):].reset_index(drop=True)
X_tr0,X_val0,_ = prepare_fold(tr0,val0,test_master)
y_val0 = y_train[int(0.8*len(tm)):]

shap_model    = models_store[1][0]
print('Computing SHAP values...')
explainer     = shap.TreeExplainer(shap_model)
shap_sample   = X_val0.sample(min(3000,len(X_val0)),random_state=SEED)
shap_vals_raw = explainer.shap_values(shap_sample)
if isinstance(shap_vals_raw,list):
    shap_vals = shap_vals_raw
else:
    shap_vals = [shap_vals_raw[:,:,i] for i in range(shap_vals_raw.shape[2])]
assert shap_vals[0].shape==shap_sample.shape, 'SHAP shape mismatch'
print(f'SHAP ready: {len(shap_vals)} classes x {shap_vals[0].shape[0]} samples x {shap_vals[0].shape[1]} features')

# Beeswarm ESI-1
print('\nSHAP Beeswarm — ESI-1 (Immediate):')
shap.summary_plot(shap_vals[0],shap_sample,max_display=15,plot_type='dot',show=False)
plt.title('SHAP — ESI-1 (Immediate) Feature Impact')
plt.tight_layout()
save_mpl('shap_beeswarm_esi1.png')

# Beeswarm ESI-3
print('\nSHAP Beeswarm — ESI-3 (Urgent):')
shap.summary_plot(shap_vals[2],shap_sample,max_display=15,plot_type='dot',show=False)
plt.title('SHAP — ESI-3 (Urgent) Feature Impact')
plt.tight_layout()
save_mpl('shap_beeswarm_esi3.png')

In [ ]:
# Global bar chart — all classes
fig,axes = plt.subplots(1,5,figsize=(22,5))
for ci,(class_idx,title) in enumerate([(0,'ESI-1'),(1,'ESI-2'),(2,'ESI-3'),(3,'ESI-4'),(4,'ESI-5')]):
    plt.sca(axes[ci])
    shap.summary_plot(shap_vals[class_idx],shap_sample,max_display=10,plot_type='bar',show=False)
    axes[ci].set_title(f'SHAP\n{title}',fontsize=10)
plt.suptitle('SHAP Global Feature Importance by ESI Class',fontsize=13,y=1.02)
plt.tight_layout()
save_mpl('shap_global_by_class.png')

In [ ]:
# Waterfall plots for 3 cases
val_preds0  = shap_model.predict(X_val0)
shap_idx_list = shap_sample.index.tolist()
cases = {
    'ESI-1 Correctly Identified':   np.where((y_val0==0)&(val_preds0==0))[0],
    'ESI-2 Dangerous Miss (→ESI-3)':np.where((y_val0==1)&(val_preds0==2))[0],
    'ESI-3 Edge Case':              np.where((y_val0==2)&(val_preds0==2))[0],
}
for name,idxs in cases.items():
    if len(idxs)==0: print(f'No patients for: {name}'); continue
    pos      = next((shap_idx_list.index(c) for c in idxs if c in shap_idx_list),0)
    true_cls = y_val0[shap_idx_list[pos]]
    sv       = shap_vals[true_cls][pos]
    bv_raw   = explainer.expected_value
    bv       = bv_raw[true_cls] if isinstance(bv_raw,(list,np.ndarray)) else bv_raw
    exp = shap.Explanation(values=sv,base_values=bv,
                           data=shap_sample.iloc[pos],
                           feature_names=shap_sample.columns.tolist())
    print(f'\n--- {name} (True: ESI-{true_cls+1}) ---')
    shap.waterfall_plot(exp,max_display=12,show=False)
    plt.title(f'SHAP Waterfall — {name}')
    plt.tight_layout()
    fname = 'shap_waterfall_' + name.lower().replace(' ','_').replace('(','').replace(')','').replace('→','to').replace('-','')[:30] + '.png'
    save_mpl(fname)

In [ ]:
# ── LIME local explanations (saved as PNG) ───────────────────────────────────
print('=== LIME LOCAL EXPLANATIONS ===')
print('LIME builds a local linear approximation around each prediction.')

lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_tr0.values,
    feature_names=X_tr0.columns.tolist(),
    class_names=[f'ESI-{i}' for i in range(1,6)],
    mode='classification',
    random_state=SEED,
    discretize_continuous=True)

shap_idx_list_lime = shap_sample.index.tolist()
for case_name, case_class in [('ESI-1 Critical', 0), ('ESI-3 Borderline', 2)]:
    idxs = np.where(y_val0 == case_class)[0]
    if len(idxs) == 0:
        continue
    pos = next((shap_idx_list_lime.index(c) for c in idxs if c in shap_idx_list_lime), None)
    if pos is None:
        continue
    row = X_val0.iloc[pos].values
    exp = lime_explainer.explain_instance(
        data_row=row,
        predict_fn=shap_model.predict_proba,
        num_features=10,
        top_labels=1)
    pred_label = int(np.argmax(shap_model.predict_proba(row.reshape(1,-1))[0]))
    fig = exp.as_pyplot_figure(label=pred_label)
    fig.set_size_inches(10, 6)
    fig.suptitle(f'LIME Local Explanation — {case_name} (Predicted ESI-{pred_label+1})',
                 fontsize=12, y=1.02)
    plt.tight_layout()
    fname = f'lime_{case_name.lower().replace(" ","_")}.png'
    save_mpl(fname)
    print(f'Saved: {fname}')


In [ ]:
# Clinical rationale generator
INTERP = {
    'news2_score':      {(0,2):'low risk',(3,4):'medium risk',(5,6):'high risk',(7,99):'very high risk'},
    'shock_index':      {(0,0.6):'normal',(0.6,1.0):'mildly elevated',(1.0,99):'critical'},
    'spo2':             {(95,100):'normal',(90,95):'borderline hypoxic',(0,90):'critically hypoxic'},
    'gcs_total':        {(13,15):'alert',(9,13):'moderate impairment',(0,9):'severe compromise'},
    'heart_rate':       {(60,100):'normal',(100,120):'tachycardic',(120,999):'severely tachycardic'},
    'systolic_bp':      {(90,140):'normal',(140,180):'hypertensive',(180,999):'hypertensive crisis',(0,90):'hypotensive'},
    'pain_score_clean': {(0,3):'mild',(4,6):'moderate',(7,10):'severe'},
    'qsofa':            {(0,0):'negative',(1,1):'1 criterion',(2,99):'POSITIVE — sepsis screen'},
}
def interp_val(feat,val):
    if feat not in INTERP:
        try: return f'{float(val):.2f}'
        except: return str(val)
    try:
        fv = float(val)
        for (lo,hi),label in INTERP[feat].items():
            if lo<=fv<=hi: return f'{fv:.1f} [{label}]'
    except Exception: pass
    return str(val)

def generate_rationale(row, sv_arr, feat_names, pred_esi, confidence):
    imp  = pd.Series(np.abs(sv_arr),index=feat_names)
    excl = [c for c in imp.index if c.startswith(('tfw_','tfc_','nlp_pca_'))]
    imp  = imp.drop(excl,errors='ignore')
    top3 = imp.nlargest(3)
    lines = []
    for feat,_ in top3.items():
        val  = row.get(feat,'N/A')
        sign = sv_arr[feat_names.index(feat)] if feat in feat_names else 0
        dir_ = 'increases severity' if sign>0 else 'decreases severity'
        lines.append(f'  {feat.replace("_"," ")}: {interp_val(feat,val)} ({dir_})')
    return (f'Predicted ESI-{pred_esi} (confidence {confidence:.1%})\n'
            f'Primary drivers:\n'+"\n".join(lines)+
            '\nADVISORY: Decision-support only — clinician judgment required.')

print('=== CLINICAL RATIONALE EXAMPLES ===')
for name,idxs in list(cases.items())[:2]:
    if len(idxs)==0: continue
    pos  = next((shap_idx_list.index(c) for c in idxs if c in shap_idx_list),0)
    row  = X_val0.iloc[pos]
    prob = shap_model.predict_proba(row.values.reshape(1,-1))[0]
    pred = int(np.argmax(prob))
    sv   = shap_vals[pred][pos]
    print(f'\n{name}:')
    print(generate_rationale(row,sv,X_val0.columns.tolist(),pred+1,prob[pred]))

In [ ]:
# ── SHAP-Based Counterfactual Analysis — "what minimal change moves ESI-3 → ESI-2?" ───
print('=== SHAP-BASED COUNTERFACTUAL ANALYSIS ===')
print('Question: What minimal vital sign changes would escalate this ESI-3 patient to ESI-2?')

esi3_idxs = np.where(y_val0 == 2)[0]
shap_idx_list = shap_sample.index.tolist()

if len(esi3_idxs) > 0:
    # Find an ESI-3 patient that is in the SHAP sample
    sample_pos = next((shap_idx_list.index(c) for c in esi3_idxs if c in shap_idx_list), None)

    if sample_pos is not None:
        patient_row = X_val0.iloc[sample_pos]
        prob_vec    = shap_model.predict_proba(patient_row.values.reshape(1,-1))[0]

        print(f'Patient: True ESI-3 | Model confidence: {prob_vec[2]:.1%} ESI-3')
        print()

        # SHAP-based counterfactual — show what drives ESI-2 for this patient
        sv_esi2 = pd.Series(shap_vals[1][sample_pos], index=X_val0.columns)
        sv_esi2 = sv_esi2.drop([c for c in sv_esi2.index
                                 if c.startswith(('tfw_','tfc_','nlp_pca_'))], errors='ignore')

        positive = sv_esi2.sort_values(ascending=False).head(5)
        negative = sv_esi2.sort_values(ascending=True).head(5)

        print('=== SHAP-BASED COUNTERFACTUAL ANALYSIS ===')
        print('Features INCREASING ESI-2 probability (already present — escalation drivers):')
        for feat, val in positive.items():
            curr_val = patient_row.get(feat, 'N/A')
            print(f'  {feat:<35} current={float(curr_val):.2f}  SHAP_ESI2={val:+.3f}')

        print()
        print('Features that if INCREASED would most raise ESI-2 probability:')
        for feat, val in negative.items():
            curr_val = patient_row.get(feat, 'N/A')
            print(f'  {feat:<35} current={float(curr_val):.2f}  SHAP_ESI2={val:+.3f}')

        print()
        print('Clinical interpretation:')
        top_escalator = positive.index[0] if len(positive) > 0 else 'news2_score'
        print(f'  If {top_escalator.replace("_"," ")} worsens further, ESI should be reassessed.')
        print('  The model flags this patient as borderline between ESI-2 and ESI-3.')

        # Save counterfactual table as PNG
        cf_data = pd.DataFrame({
            'Feature': list(positive.index) + list(negative.index),
            'Current Value': [float(patient_row.get(f,0)) for f in list(positive.index)+list(negative.index)],
            'SHAP for ESI-2': list(positive.values) + list(negative.values),
            'Direction': ['escalates →ESI-2']*5 + ['reduces ESI-2 prob']*5
        })
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.axis('off')
        tbl = ax.table(cellText=cf_data.round(3).values, colLabels=cf_data.columns,
                       cellLoc='center', loc='center')
        tbl.auto_set_font_size(False); tbl.set_fontsize(9)
        tbl.auto_set_column_width(col=list(range(len(cf_data.columns))))
        ax.set_title('Counterfactual Analysis — ESI-3 Patient: Factors Driving ESI-2 Escalation',
                     fontsize=11, pad=20)
        plt.tight_layout()
        save_mpl('shap_counterfactual_esi3_to_esi2.png')
    else:
        print('No ESI-3 patients found in SHAP sample.')
else:
    print('No ESI-3 patients in validation set.')


---
## Section 11 — Ablation Study

Rather than removing features one at a time, this section builds up the feature set **cumulatively** — vitals only → + demographics → + comorbidities → + clinical risk scores/safety flags → + keyword-tier NLP → + full semantic NLP (PCA embeddings) — and re-scores a fast 3-fold LightGBM (`ablation_qwk`) at each stage. The resulting QWK at each step shows how much each *additional* feature group contributes on top of what came before, rather than relying on intuition about what should matter clinically. A bootstrap test (1,000 resamples) then directly compares the vitals-only model against the full stacked ensemble and reports a one-sided p-value, so the improvement from all the added engineering can be called statistically significant (or not) rather than just numerically larger.


In [ ]:
def ablation_qwk(name, feat_list, n_folds=3):
    feat_list = [f for f in feat_list if f in train_master.columns]
    X = train_master[feat_list].copy().fillna(0)
    scores = []
    for tr_i,val_i in StratifiedKFold(n_folds,shuffle=True,random_state=SEED).split(X,y_train):
        m = lgb.LGBMClassifier(objective='multiclass',num_class=5,n_estimators=200,
                                learning_rate=0.1,num_leaves=31,class_weight='balanced',
                                random_state=SEED,verbose=-1,n_jobs=-1)
        m.fit(X.iloc[tr_i],y_train[tr_i])
        scores.append(cohen_kappa_score(y_train[val_i],m.predict(X.iloc[val_i]),weights='quadratic'))
    q=np.mean(scores); print(f'  {name:<50} QWK: {q:.4f}'); return q

vitals   = [c for c in ['heart_rate','systolic_bp','diastolic_bp','respiratory_rate',
                          'temperature_c','spo2','gcs_total','news2_score','shock_index',
                          'bmi','pain_score_clean'] if c in train_master.columns]
demo     = vitals+[c for c in ['age','elderly','pediatric','very_elderly'] if c in train_master.columns]
comorb   = demo+[c for c in train_master.columns if c.startswith('hx_')]+           [c for c in ['comorbidity_count','high_comorbidity'] if c in train_master.columns]
clinical = comorb+[c for c in ['qsofa','sirs_score','gcs_severe','gcs_moderate','spo2_critical',
                                 'spo2_concerning','sbp_hypotensive','sbp_hypertensive','pain_severe',
                                 'mental_status_encoded','flag_stemi','flag_sepsis','flag_shock',
                                 'any_safety_flag','vitals_missing_count'] if c in train_master.columns]
keywords = clinical+[c for c in train_master.columns if c.startswith('kw_')]+           [c for c in ['chief_complaint_len'] if c in train_master.columns]
with_nlp = keywords+[c for c in NLP_PCA_COLS if c in train_master.columns]

print('=== ABLATION STUDY ===')
abl_results = []
for name,feats in [('1. Vitals only',vitals),('2. + Demographics',demo),
                    ('3. + Comorbidities',comorb),('4. + Clinical flags',clinical),
                    ('5. + Keyword NLP',keywords),('6. + Semantic NLP (full)',with_nlp)]:
    abl_results.append({'Feature Set':name,'QWK':round(ablation_qwk(name,feats),4)})

abl_df = pd.DataFrame(abl_results)
fig = px.bar(abl_df,x='QWK',y='Feature Set',orientation='h',text='QWK',color='QWK',
             color_continuous_scale='Greens',title='Ablation Study — Cumulative Feature Contribution')
fig.update_traces(texttemplate='%{text:.4f}',textposition='outside')
fig.update_layout(height=420)
save_fig(fig,'ablation_study.png')
# ── Statistical significance of feature groups ────────────────────────────────
from scipy import stats
print('\n=== STATISTICAL SIGNIFICANCE OF FEATURE ADDITIONS ===')
print('McNemar test: is the improvement from adding each feature group significant?')
print('Comparing full model (with NLP) vs vitals-only model on OOF predictions.')
print()

# Use bootstrap resampling to test significance
vitals_feats_test = [c for c in vitals if c in train_master.columns]
X_vitals = train_master[vitals_feats_test].fillna(0)
oof_vitals = np.zeros(len(y_train), dtype=int)
for tr_i, val_i in StratifiedKFold(3,shuffle=True,random_state=SEED).split(X_vitals,y_train):
    m_v = lgb.LGBMClassifier(objective='multiclass',num_class=5,n_estimators=200,
                               class_weight='balanced',random_state=SEED,verbose=-1,n_jobs=-1)
    m_v.fit(X_vitals.iloc[tr_i], y_train[tr_i])
    oof_vitals[val_i] = m_v.predict(X_vitals.iloc[val_i])

qwk_vitals = cohen_kappa_score(y_train, oof_vitals, weights='quadratic')
qwk_full   = stack_qwk

# Bootstrap test
diffs = []
np.random.seed(SEED)
for _ in range(1000):
    idx_b = np.random.choice(len(y_train), len(y_train), replace=True)
    try:
        q_v = cohen_kappa_score(y_train[idx_b], oof_vitals[idx_b], weights='quadratic')
        q_f = cohen_kappa_score(y_train[idx_b], stack_oof[idx_b],  weights='quadratic')
        diffs.append(q_f - q_v)
    except Exception:
        pass

p_val = np.mean(np.array(diffs) <= 0)  # fraction of times full model is NOT better
print(f'Vitals-only QWK:   {qwk_vitals:.4f}')
print(f'Full ensemble QWK: {qwk_full:.4f}')
print(f'Improvement:       +{qwk_full-qwk_vitals:.4f} QWK points')
print(f'Bootstrap p-value: {p_val:.4f} (H0: full model ≤ vitals model)')
if p_val < 0.05:
    print('Result: STATISTICALLY SIGNIFICANT improvement (p<0.05)')
else:
    print('Result: Not statistically significant at p<0.05')


---
## Section 12 — Survival Analysis

ESI predicts who should be seen first, but acuity should also predict downstream clinical course. Kaplan–Meier curves of time-to-admission (`ed_los_hours`, capped at 24h), stratified by true ESI level, test whether higher-acuity patients are admitted faster — a face-validity check that goes beyond the static QWK metric. A Cox proportional-hazards model (`CoxPHFitter`) is then fit on the same cohort using vitals and risk scores (age, HR, SBP, SpO2, GCS, NEWS2, shock index, comorbidity count, qSOFA) as covariates, producing hazard ratios that quantify *how much* each factor accelerates time-to-admission — directly relevant to the deterioration-risk use case named in the competition brief.


In [ ]:
if 'disposition' in train_master.columns and 'ed_los_hours' in train_master.columns:
    surv = train_master.copy()
    surv['event']    = surv['disposition'].astype(str).str.lower().isin(
        ['admitted','icu','inpatient','hospitalized','admit']).astype(int)
    surv['duration'] = surv['ed_los_hours'].clip(0.1,24).fillna(surv['ed_los_hours'].median())
    print(f'Cohort: {len(surv):,} | Event rate: {surv["event"].mean():.1%}')
    fig,ax = plt.subplots(figsize=(10,5))
    for esi in range(1,6):
        sub = surv[surv['triage_acuity']==esi]
        if len(sub)<10: continue
        KaplanMeierFitter().fit(sub['duration'],event_observed=sub['event'],
                                label=f'ESI-{esi} (n={len(sub):,})').plot_survival_function(
            ax=ax,color=['#d32f2f','#f57c00','#fbc02d','#388e3c','#1976d2'][esi-1],ci_show=True)
    ax.set_title('Kaplan-Meier — Time to Admission by ESI Level')
    ax.set_xlabel('Hours in ED'); ax.set_ylabel('P(not yet admitted)')
    plt.tight_layout(); save_mpl('survival_kaplan_meier.png')
    cox_cols = [c for c in ['age','heart_rate','systolic_bp','spo2','gcs_total',
                              'news2_score','shock_index','comorbidity_count','qsofa'] if c in surv.columns]
    cox_df = surv[cox_cols+['duration','event']].dropna()
    cph = CoxPHFitter(penalizer=0.1)
    cph.fit(cox_df,duration_col='duration',event_col='event')
    print('\nCox PH Hazard Ratios:')
    cph.print_summary()
else:
    print('disposition/ed_los_hours not in train_master.')

---
## Section 13 — Bias & Fairness Audit

Undertriage — predicting a less urgent ESI than the true label — is the failure mode with the worst clinical consequences, and prior literature documents it is not evenly distributed across patient groups (Obermeyer et al., 2019). This section runs four checks, all restricted to the highest-acuity cohort (ESI-1/2) where undertriage is most dangerous:
- **Subgroup undertriage rates** by sex, age group, insurance type, and language, plus a heatmap visualization.
- **`fairlearn` metrics** — per-group false-negative rate via `MetricFrame`, and the equalized-odds difference for a binarized "high-acuity vs not" target, by sex and insurance type.
- **Equity-adjusted QWK** — overall model QWK recomputed within each demographic subgroup, plotted against the overall QWK as a reference line, so any subgroup performing meaningfully worse than average is visually obvious.
- **Temporal bias and an odds-ratio forest plot** — undertriage rate by arrival hour, plus a logistic regression (`statsmodels.Logit`) of undertriage on demographics and arrival factors, reported as odds ratios with 95% CIs and significance-colored markers.

A model that performs well in aggregate but undertriages a specific population is not safe to deploy regardless of its overall score — this is the audit that would need to clear before any pilot of the kind described in the competition brief.


In [ ]:
bias_df = train_master.copy()
bias_df['model_pred_esi'] = stack_oof+1
bias_df['undertriaged']   = (bias_df['model_pred_esi']>bias_df['triage_acuity']).astype(int)
high_ac = bias_df[bias_df['triage_acuity']<=2]
print(f'=== UNDERTRIAGE — ESI-1/2 PATIENTS (n={len(high_ac):,}) ===')
print(f'Overall undertriage rate: {high_ac["undertriaged"].mean():.1%}\n')
for col in ['sex','age_group','insurance_type','language']:
    if col not in bias_df.columns: continue
    rates = high_ac.groupby(bias_df.loc[high_ac.index,col])['undertriaged'].agg(['mean','count'])
    rates.columns=['rate','n']; rates['rate_pct']=(rates['rate']*100).round(1)
    print(f'--- {col} ---'); print(rates.sort_values('rate',ascending=False).to_string()); print()

In [ ]:
# Equity heatmap
demo_cols_bias = [c for c in ['sex','age_group','insurance_type'] if c in bias_df.columns]
if demo_cols_bias:
    fig,axes = plt.subplots(1,len(demo_cols_bias),figsize=(5*len(demo_cols_bias),5))
    if len(demo_cols_bias)==1: axes=[axes]
    for ax,col in zip(axes,demo_cols_bias):
        rates = high_ac.groupby(bias_df.loc[high_ac.index,col])['undertriaged'].mean()*100
        data  = pd.DataFrame({'Undertriage Rate (%)':rates})
        sns.heatmap(data,annot=True,fmt='.1f',cmap='RdYlGn_r',ax=ax,
                    vmin=0,vmax=40,cbar_kws={'label':'Undertriage Rate (%)'})
        ax.set_title(f'By {col}\n(ESI-1/2 only)')
    plt.suptitle('Equity Analysis — Undertriage Rate by Demographic Group',fontsize=13,y=1.02)
    plt.tight_layout(); save_mpl('equity_undertriage_heatmap.png')

# fairlearn
for scol in ['sex','insurance_type']:
    if scol not in bias_df.columns: continue
    y_tb = (bias_df['triage_acuity']<=2).astype(int)
    y_pb = (bias_df['model_pred_esi']<=2).astype(int)
    mf   = MetricFrame(metrics={'fnr':false_negative_rate,'acc':lambda yt,yp:float(np.mean(yt==yp))},
                       y_true=y_tb,y_pred=y_pb,sensitive_features=bias_df[scol].astype(str))
    eod  = equalized_odds_difference(y_tb,y_pb,sensitive_features=bias_df[scol].astype(str))
    print(f'\nfairlearn [{scol}]:'); print(mf.by_group)
    print(f'Equalized Odds Difference: {eod:.4f}')

In [ ]:
# Equity kappa by group
if 'sex' in bias_df.columns:
    eq_rows = []
    for grp in bias_df['sex'].dropna().unique():
        mask = bias_df['sex']==grp
        qwk_g = cohen_kappa_score(y_train[mask],stack_oof[mask],weights='quadratic')
        eq_rows.append({'Group':f'sex={grp}','QWK':round(qwk_g,4),'n':int(mask.sum())})
    if 'age_group' in bias_df.columns:
        for grp in bias_df['age_group'].dropna().unique():
            mask = bias_df['age_group']==grp
            if mask.sum()<50: continue
            qwk_g = cohen_kappa_score(y_train[mask],stack_oof[mask],weights='quadratic')
            eq_rows.append({'Group':f'age={grp}','QWK':round(qwk_g,4),'n':int(mask.sum())})
    eq_df = pd.DataFrame(eq_rows).sort_values('QWK')
    fig = px.bar(eq_df,x='QWK',y='Group',orientation='h',text='QWK',color='QWK',
                 color_continuous_scale='RdYlGn',
                 title='QWK by Demographic Group — lower = worse clinical performance for that group')
    fig.add_vline(x=stack_qwk,line_dash='dash',line_color='black',
                  annotation_text=f'Overall QWK={stack_qwk:.3f}')
    fig.update_traces(texttemplate='%{text:.4f}',textposition='outside')
    fig.update_layout(height=max(350,len(eq_df)*40))
    save_fig(fig,'equity_kappa_by_group.png')

# Temporal bias
if 'arrival_hour' in bias_df.columns:
    tb  = high_ac.groupby(bias_df.loc[high_ac.index,'arrival_hour'])['undertriaged'].mean()*100
    fig = px.bar(x=tb.index,y=tb.values,color=tb.values,color_continuous_scale='RdYlGn_r',
                 title='Undertriage Rate by Hour (ESI-1/2 patients)',
                 labels={'x':'Hour','y':'Undertriage Rate (%)'})
    fig.update_layout(height=340); save_fig(fig,'equity_temporal_bias.png')

# Odds ratio forest
lr_base = [c for c in ['elderly','pediatric','weekend','night_shift','high_risk_arrival'] if c in bias_df.columns]
dum_frames=[]
for col in ['sex','insurance_type']:
    if col not in bias_df.columns: continue
    dums = pd.get_dummies(bias_df[col],prefix=col,drop_first=True,dtype=float)
    new_ = [c for c in dums.columns if c not in bias_df.columns]
    if new_: dum_frames.append(dums[new_])
if dum_frames: bias_df = pd.concat([bias_df]+dum_frames,axis=1)
dum_names = [c for df_ in dum_frames for c in df_.columns]
inp_cols  = [c for c in lr_base+dum_names if c in bias_df.columns]
if inp_cols:
    X_lr = sm.add_constant(bias_df[inp_cols].fillna(0).astype(float))
    X_lr = X_lr.loc[:,~X_lr.columns.duplicated()]; X_lr = X_lr.loc[:,X_lr.var()>0]
    y_lr = high_ac.reindex(bias_df.index)['undertriaged'].fillna(0).astype(float)
    try:
        lr_res  = sm.Logit(y_lr,X_lr).fit(method='lbfgs',maxiter=500,disp=0)
        coef_df = pd.DataFrame({'OR':np.exp(lr_res.params),'CI_lo':np.exp(lr_res.conf_int()[0]),
                                  'CI_hi':np.exp(lr_res.conf_int()[1]),'p':lr_res.pvalues}
                               ).drop('const',errors='ignore').sort_values('OR',ascending=False)
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=coef_df['OR'],y=coef_df.index,mode='markers',
                                 error_x=dict(type='data',symmetric=False,
                                              array=coef_df['CI_hi']-coef_df['OR'],
                                              arrayminus=coef_df['OR']-coef_df['CI_lo']),
                                 marker=dict(size=10,color=['#d32f2f' if p<0.05 else '#1976d2' for p in coef_df['p']])))
        fig.add_vline(x=1.0,line_dash='dash',line_color='gray',annotation_text='OR=1.0')
        fig.update_layout(title='Forest Plot — Odds Ratios for Undertriage Risk (red=p<0.05)',height=420)
        save_fig(fig,'equity_odds_ratio_forest.png')
    except Exception as e:
        print(f'Logistic regression note: {e}')

---
## Section 14 — Responsible AI: Safety Layer + Model Card

Two deployment-facing artifacts close the gap between a Kaggle notebook and something a hospital could plausibly pilot. First, a **safety layer**: predictions whose maximum class probability falls below an abstention threshold (0.45) are flagged for `manual_review`; separately, any patient who trips one of the five rule-based safety flags from Section 5 *and* was predicted ESI-3 or less urgent is automatically escalated to ESI-2 and flagged for review, regardless of what the model's probabilities said — the rule-based check is a hard backstop, not just an input feature. Second, a formal **model card** documents architecture, intended use, key limitations (the Section 4 shortcut, single-site calibration, ESI label noise), and safeguards in the standard format clinicians and IRBs expect to see before any pilot conversation begins.


In [ ]:
ABSTENTION_THRESHOLD = 0.45
test_max_prob   = stack_test_prob.max(axis=1)
test_pred_final = apply_thresholds(stack_test_prob, OPTIMAL_THRESHOLDS)  # already 1-indexed (1..5)
abstain_mask    = test_max_prob < ABSTENTION_THRESHOLD
test_master['final_esi']      = test_pred_final.copy()
test_master['manual_review']  = abstain_mask

flag_cols = [c for c in ['flag_stemi','flag_sepsis','flag_stroke','flag_resp_failure','flag_shock']
             if c in test_master.columns]
if flag_cols:
    any_flag = test_master[flag_cols].sum(axis=1)>0
    escalate = any_flag & (test_master['final_esi']>=3)
    test_master.loc[escalate,'final_esi']    = 2
    test_master.loc[escalate,'manual_review'] = True
    print(f'Safety escalations: {escalate.sum():,} patients → ESI-2')
    for fc in flag_cols:
        n=test_master[fc].sum()
        if n>0: print(f'  {fc}: {n:,} flagged')

print('\nFinal ESI distribution:')
for esi,cnt in test_master['final_esi'].value_counts().sort_index().items():
    bar=chr(9608)*int(cnt/200)
    print(f'  ESI-{esi}: {cnt:>5,} ({cnt/len(test_master)*100:.1f}%) {bar}')
print(f'\nCalibration (Brier Score):')
for ci,cn in enumerate(['ESI-1','ESI-2','ESI-3','ESI-4','ESI-5']):
    bs=brier_score_loss((y_train==ci).astype(int),oof_prob_stack[:,ci])
    print(f'  {cn}: {bs:.4f}')

In [ ]:
from IPython.display import Markdown,display
display(Markdown("""
# MODEL CARD — TriageGeist ESI Prediction System v1.0

| Attribute | Value |
|-----------|-------|
| Architecture | 4-model stacked ensemble (XGBoost GPU + LightGBM CPU + CatBoost GPU + XGBoost tabular) |
| Task | 5-class ordinal classification — ESI 1–5 |
| Primary metric | Quadratic Weighted Kappa (QWK) |
| Training data | Triagegeist synthetic dataset — 80,000 patients |

## Intended Use
Clinical decision **support** — advisory second opinion for triage nurses. Not a replacement for clinical judgment.

## Key Limitations
1. **Synthetic shortcut** — Section 4 GroupKFold audit proves text memorisation. Tabular-only QWK = honest benchmark.
2. **Single-source** — MIMIC-IV-ED calibration. Community ED generalisability unknown.
3. **Label noise** — ESI is human-assigned (κ≈0.68). Model inherits human variability.

## Safeguards
- Confidence abstention: max_prob < 0.45 → mandatory senior review flag
- Conservative safety flags: NaN-safe, only extreme measured vitals escalate
- Full 5-class ESI distribution validated before submission
- Demographic fairness audit across sex, age, insurance, language (Section 13)
"""))

---
## Section 15 — Save All Artifacts + Final Submission

Final predictions are validated against five separate checks (correct row count, ESI values in 1–5, all five classes represented, no NaNs, and a sanity check that train/test class-distribution shift stays under 25 points) before `submission.csv` is written, alongside a QWK-stamped duplicate filename and a full OOF-predictions CSV with per-class probabilities for later auditing. Beyond the competition submission, the notebook also persists: a shallow (`max_depth=3`) decision tree trained on 10 core clinical features as a printable "bedside card," a `triage_rules.txt` export combining that tree's plain-text rules with the hard-coded safety-flag thresholds and the top-10 SHAP features for ESI-1, and the trained meta-model, NLP PCA transform, optimal thresholds, and NLP risk classifier as pickled/`.npy` artifacts — everything a reviewer or downstream team would need to reuse the pipeline without retraining from scratch.


In [ ]:
# ── Validate final predictions ────────────────────────────────────────────────
final_preds = test_master['final_esi'].values.astype(int)

assert len(final_preds)  == len(sample_sub), f'Row count: {len(final_preds)} != {len(sample_sub)}'
assert final_preds.min() >= 1,               f'ESI below 1: {final_preds.min()}'
assert final_preds.max() <= 5,               f'ESI above 5: {final_preds.max()}'
assert len(np.unique(final_preds)) == 5,     f'Missing ESI classes: {sorted(np.unique(final_preds))}'
assert not np.isnan(final_preds.astype(float)).any(), 'NaN in predictions'

submission = pd.DataFrame({'patient_id':test['patient_id'].values,'triage_acuity':final_preds})
assert list(submission.columns) == list(sample_sub.columns), 'Column mismatch'

train_dist = train['triage_acuity'].value_counts(normalize=True).sort_index()
test_dist  = submission['triage_acuity'].value_counts(normalize=True).reindex(train_dist.index,fill_value=0)
max_shift  = (train_dist-test_dist).abs().max()
assert max_shift < 0.25, f'Distribution shift too large: {max_shift:.3f}'

print('=== FINAL SUBMISSION VALIDATION ===')
print(f'{"ESI":<6}{"Train %":<14}{"Test %":<14}{"Delta"}')
print('-'*44)
for esi in range(1,6):
    tr_p = train_dist.get(esi,0)*100; te_p = test_dist.get(esi,0)*100
    flag = '  ← LARGE SHIFT' if abs(te_p-tr_p)>10 else ''
    print(f'{esi:<6}{tr_p:<14.1f}{te_p:<14.1f}{te_p-tr_p:+.1f}{flag}')
print(f'\nMax distribution shift: {max_shift:.3f} (threshold: 0.25)')
print('\n=== RESULTS TABLE ===')
rows=[]
for i,n in enumerate(MODEL_NAMES):
    rows.append({'Model':n,'OOF QWK':f'{np.mean(qwk_scores[i]):.4f}','Std':f'{np.std(qwk_scores[i]):.4f}'})
rows.append({'Model':'Stacked Ensemble','OOF QWK':f'{stack_qwk:.4f}','Std':'-'})
rows.append({'Model':'+ Threshold Opt (FINAL)','OOF QWK':f'{opt_qwk:.4f}','Std':'-'})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# ── Save submission CSV ───────────────────────────────────────────────────────
submission.to_csv(OUT+'submission.csv',index=False)
qwk_fname = f'submission_qwk_{opt_qwk:.4f}.csv'.replace('.','p',1).replace('.','_')
submission.to_csv(OUT+qwk_fname,index=False)
print(f'Saved: submission.csv ({len(submission):,} rows)')
print(f'Saved: {qwk_fname}')

# ── OOF predictions CSV ───────────────────────────────────────────────────────
oof_df = pd.DataFrame({'patient_id':train['patient_id'],'true_esi':y_train+1,
                        'pred_esi':stack_oof+1,'confidence':oof_prob_stack.max(axis=1)})
for i in range(5):
    oof_df[f'prob_esi{i+1}'] = oof_prob_stack[:,i]
oof_df.to_csv(OUT+'oof_predictions.csv',index=False)
print(f'Saved: oof_predictions.csv ({len(oof_df):,} rows)')

In [ ]:
# ── Bedside decision tree card ────────────────────────────────────────────────
dt_feats = [c for c in ['news2_score','gcs_total','pain_score_clean','shock_index',
                          'spo2','heart_rate','respiratory_rate','qsofa','comorbidity_count','age']
            if c in train_master.columns]
X_dt = train_master[dt_feats].fillna(0).values
dt   = DecisionTreeClassifier(max_depth=3,class_weight='balanced',random_state=SEED)
dt.fit(X_dt, y_train+1)
fig,ax = plt.subplots(figsize=(22,9))
plot_tree(dt,feature_names=dt_feats,class_names=[f'ESI-{i}' for i in range(1,6)],
          filled=True,rounded=True,fontsize=9,ax=ax,impurity=False,proportion=True)
ax.set_title(f'Rapid Triage Bedside Card — Decision Tree (depth=3)\n'
             f'Threshold values derived from {len(train_master):,} ED patient visits',fontsize=12)
plt.tight_layout(); save_mpl('bedside_card_tree.png')

In [ ]:
# ── Triage rules text file ────────────────────────────────────────────────────
rules_txt  = export_text(dt,feature_names=dt_feats)
shap_imp   = pd.Series(np.abs(shap_vals[0]).mean(axis=0),index=shap_sample.columns)
shap_imp   = shap_imp.drop([c for c in shap_imp.index if c.startswith(('tfw_','tfc_','nlp_pca_'))],errors='ignore')
top10_shap = shap_imp.nlargest(10)

content = f"""TRIAGEGEIST — EXTRACTED TRIAGE RULES
Stacked ensemble OOF QWK: {stack_qwk:.4f} | Optimized: {opt_qwk:.4f}
Training set: {len(train_master):,} patients | Test set: {len(test_master):,} patients

DECISION TREE RULES (depth=3, interpretable for clinical bedside use):
{rules_txt}

CLINICAL SAFETY OVERRIDES (rule-based, NaN-safe):
  flag_stemi:        age>=45 AND HR>120 AND SBP<85       -> escalate to ESI-2
  flag_sepsis:       (temp>38.8 OR temp<35.5) AND HR>105 AND RR>24  -> escalate to ESI-2
  flag_stroke:       GCS<11 AND age>60                   -> escalate to ESI-2
  flag_resp_failure: SpO2<85 AND RR>30                   -> escalate to ESI-2
  flag_shock:        Shock Index > 1.4                   -> escalate to ESI-2
  (NaN values filled with normal physiological values — missing data never triggers a flag)

TOP 10 SHAP FEATURES FOR ESI-1 PREDICTION:
"""
for feat,val in top10_shap.items():
    content += f'  {feat:<40} mean|SHAP| = {val:.4f}\n'

content += f"""
SUBMISSION DISTRIBUTION:
"""
for esi,cnt in submission['triage_acuity'].value_counts().sort_index().items():
    content += f'  ESI-{esi}: {cnt:,} ({cnt/len(submission)*100:.1f}%)\n'

with open(OUT+'triage_rules.txt','w') as f:
    f.write(content)
print('Saved: triage_rules.txt')

In [ ]:
# ── Model artifacts ──────────────────────────────────────────────────────────
with open(OUT+'meta_model.pkl','wb') as f:
    pickle.dump(meta_model,f)
with open(OUT+'nlp_pca.pkl','wb') as f:
    pickle.dump(pca,f)
np.save(OUT+'optimal_thresholds.npy',OPTIMAL_THRESHOLDS)
with open(OUT+'lr_nlp_classifier.pkl','wb') as f:
    pickle.dump(lr_nlp,f)
print('Saved: meta_model.pkl')
print('Saved: nlp_pca.pkl')
print('Saved: optimal_thresholds.npy')
print('Saved: lr_nlp_classifier.pkl')

In [ ]:
# ── Final output summary ──────────────────────────────────────────────────────
print('\n=== ALL OUTPUT FILES ===')
output_files = sorted(os.listdir(OUT))
for f in output_files:
    size = os.path.getsize(OUT+f)
    unit = 'KB' if size<1e6 else 'MB'
    size_str = f'{size/1024:.0f} {unit}' if size<1e6 else f'{size/1e6:.1f} {unit}'
    print(f'  {f:<50} {size_str}')

print(f'\n=== COMPETITION SUBMISSION READY ===')
print(f'Primary file: submission.csv ({len(submission):,} rows)')
print(f'ESI classes:  {sorted(submission["triage_acuity"].unique())}')
print(f'OOF QWK:      {stack_qwk:.4f}')
print(f'Optimized QWK:{opt_qwk:.4f}')
print(f'\nAll assertions passed. Ready to submit.')

---
## Section 16 — Limitations, Recommendations & References

### Honest Limitations
1. **Synthetic data shortcut** — Section 4 GroupKFold audit quantifies the text memorisation gap. Teams reporting QWK > 0.95 from text features are reporting a data artifact. Tabular-only QWK is the honest clinical benchmark.
2. **Single-source** — Calibrated to MIMIC-IV-ED. Community ED generalisability unknown.
3. **Label noise** — ESI is human-assigned (κ ≈ 0.68). Model inherits human bias.
4. **Language bias** — NLP may underperform for non-English complaints.
5. **ESI-1 rarity** — ~4% of training cases. Safety flags provide rule-based backup.

### Clinical Deployment Recommendations
1. Advisory only — clinician override always logged
2. Site-level prospective validation before activation (minimum 500 patients)
3. Fairness re-audit every 6 months
4. Distribution shift monitoring weekly in production

### References
1. Gilboy et al. (2012). ESI Version 4. AHRQ.
2. Worster et al. (2004). ESI inter-rater reliability. Acad Emerg Med 11(11).
3. Obermeyer et al. (2019). Racial bias in healthcare algorithms. Science 366.
4. Johnson et al. (2023). MIMIC-IV. Scientific Data 10.
5. Angelopoulos & Bates (2021). Conformal Prediction. arXiv:2107.07511.
6. Lundberg & Lee (2017). SHAP. NeurIPS 30.
7. Reimers & Gurevych (2019). Sentence-BERT. EMNLP.
8. Ng et al. (2010). Taiwan Triage validation. Emerg Med J 28.

### Reproducibility
- Seed: 2025 | Kaggle GPU T4×2 | Python 3.12
- Runtime: ~30–45 minutes end-to-end
- All cells run top-to-bottom without errors or warnings
10. Ribeiro MT, Singh S, Guestrin C (2016). LIME — explaining classifier predictions. *KDD*.